In [1]:
!pip -q install FlagEmbedding faiss-cpu


  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 247.7/247.7 kB 6.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 81.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 866.1/866.1 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 149.7/149.7 kB 11.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.6/45.6 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 60.6 MB/s eta 0:00:00


In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HUGGING_FACE_HUB_TOKEN")

## v4-safe changes

This version keeps the original 0.07802 pipeline as the backbone and adds only conservative citation-pattern rescue: no broad legal semantic expansion, no multi-query dense retrieval, and only a tiny verifier/tie-breaker.


## v6-tail-rules modifications

Built on the original v5-safe notebook. The dense/BGE backbone and feature pipeline are left unchanged.
Only the final safe-tail layer is strengthened: same Art/Law + Abs verifier, adjacent-article tail-only boost, confident co-citation tail-only rescue, and slightly wider safe tail settings.

Goal: improve citation-level ordering without letting rule expansion pollute the head ranking.


In [3]:
import os
import re
import gc
import numpy as np
import pandas as pd
import torch
import faiss

from tqdm.auto import tqdm
from collections import defaultdict, Counter
from FlagEmbedding import BGEM3FlagModel
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity


In [4]:
BASE_PATH = "/kaggle/input/competitions/llm-agentic-legal-information-retrieval"

train = pd.read_csv(f"{BASE_PATH}/train.csv")
val = pd.read_csv(f"{BASE_PATH}/val.csv")
test = pd.read_csv(f"{BASE_PATH}/test.csv")
laws = pd.read_csv(f"{BASE_PATH}/laws_de.csv")
court = pd.read_csv(f"{BASE_PATH}/court_considerations.csv")
sample_sub = pd.read_csv(f"{BASE_PATH}/sample_submission.csv")

print("train :", train.shape)
print("val   :", val.shape)
print("test  :", test.shape)
print("laws  :", laws.shape)
print("court :", court.shape)

print("\ntrain columns:", train.columns.tolist())
print("val columns  :", val.columns.tolist())
print("test columns :", test.columns.tolist())
print("laws columns :", laws.columns.tolist())
print("court columns:", court.columns.tolist())
print("sample columns:", sample_sub.columns.tolist())

train : (1139, 3)
val   : (10, 3)
test  : (40, 2)
laws  : (175933, 3)
court : (2476315, 2)

train columns: ['query_id', 'query', 'gold_citations']
val columns  : ['query_id', 'query', 'gold_citations']
test columns : ['query_id', 'query']
laws columns : ['citation', 'text', 'title']
court columns: ['citation', 'text']
sample columns: ['query_id', 'predicted_citations']


In [5]:
def normalize_text(text):
    text = str(text).lower()
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def split_citations(s):
    if pd.isna(s) or str(s).strip() == "":
        return []
    return [x.strip() for x in str(s).split(";") if x.strip()]

def f1_score_set(pred, gold):
    pred_set = set(pred)
    gold_set = set(gold)

    if len(pred_set) == 0 and len(gold_set) == 0:
        return 1.0
    if len(pred_set) == 0 or len(gold_set) == 0:
        return 0.0

    tp = len(pred_set & gold_set)
    if tp == 0:
        return 0.0

    precision = tp / len(pred_set)
    recall = tp / len(gold_set)
    return 2 * precision * recall / (precision + recall)

def minmax(x):
    x = np.array(x, dtype=np.float32)
    if x.max() - x.min() < 1e-8:
        return np.zeros_like(x)
    return (x - x.min()) / (x.max() - x.min())

In [6]:
all_citations = set()

for df in [train, val]:
    for s in df["gold_citations"].fillna(""):
        all_citations.update(split_citations(s))

all_citations = sorted(list(all_citations))
gold_citation_set = set(all_citations)

print("Total gold citations:", len(all_citations))
print(all_citations[:20])

Total gold citations: 2878
['1B_15/2023 E. 3.1', '1B_192/2022 E. 4.1.2', '1B_195/2022 E. 2.2.1', '1B_210/2023 E. 4.1', '1B_211/2017 E. 2.1', '1B_28/2022 E. 4.1', '1B_357/2022 E. 3.1', '1B_536/2018 E. 5.1', '1B_572/2021 E. 2.1', '1B_581/2022 E. 2.1.2', '1B_88/2022 E. 2.1', '1B_90/2021 E. 2.1', '1B_90/2021 E. 2.4', '2C_501/2020 E. 5.1', '4A_379/2016 E. 3.3.1', '4A_42/2015 E. 5.5', '4A_42/2015 E. 6.3', '4A_42/2015 E. 6.6', '5A_561/2020 E. 5.1.1', '5A_954/2015 E. 3.3']


In [7]:
laws = laws[["citation", "text"]].dropna().copy()
court = court[["citation", "text"]].dropna().copy()

laws["citation"] = laws["citation"].astype(str).str.strip()
court["citation"] = court["citation"].astype(str).str.strip()

laws["text"] = laws["text"].astype(str).str.slice(0, 1800)
court["text"] = court["text"].astype(str).str.slice(0, 1800)

# ------------------------------------------------------------
# Keep full cleaned copies before the high-precision gold filter.
# BGE-M3 still encodes the small gold-core corpus for speed/memory,
# but feature/citation expansion can now recover law citations that
# never appeared in train/val gold labels.
# ------------------------------------------------------------
laws_all = laws.drop_duplicates(subset=["citation", "text"]).reset_index(drop=True)
court_all = court.drop_duplicates(subset=["citation", "text"]).reset_index(drop=True)

# Core corpus for dense BGE encoding: keep the original high-scoring gold-core idea.
laws = laws_all[laws_all["citation"].isin(gold_citation_set)].copy()
court = court_all[court_all["citation"].isin(gold_citation_set)].copy()

laws["source"] = "laws_core"
court["source"] = "court_core"

corpus = pd.concat([laws, court], ignore_index=True)
corpus = corpus.drop_duplicates(subset=["citation", "text"]).reset_index(drop=True)

# Feature layer uses citation-level court aggregation.
# Keep court feature conservative: filtered court is less noisy.
court_full = (
    court.groupby("citation", as_index=False)["text"]
    .agg(lambda x: " ".join(x.astype(str).head(3)))
)
court_full["text"] = court_full["text"].astype(str).str.slice(0, 3000)

# Full court aggregation is kept only for optional conservative direct-case expansion/debugging.
court_all_full = (
    court_all.groupby("citation", as_index=False)["text"]
    .agg(lambda x: " ".join(x.astype(str).head(2)))
)
court_all_full["text"] = court_all_full["text"].astype(str).str.slice(0, 2200)

print("full laws     :", laws_all.shape)
print("full court    :", court_all.shape)
print("filtered laws :", laws.shape)
print("filtered court:", court.shape)
print("corpus:", corpus.shape)
print("court_full:", court_full.shape)
print("court_all_full:", court_all_full.shape)

corpus.head()


full laws     : (175933, 2)
full court    : (2476013, 2)
filtered laws : (1958, 3)
filtered court: (156, 3)
corpus: (2114, 3)
court_full: (153, 2)
court_all_full: (1985178, 2)


,citation,text,source
0,Art. 1a Abs. 1 AHVG,1 Versichert nach diesem Gesetz sind:11a.12 di...,laws_core
1,Art. 3 Abs. 2 AHVG,2 Von der Beitragspflicht sind befreit:a.32 di...,laws_core
2,Art. 5 Abs. 1 AHVG,1 Vom Einkommen aus unselbständiger Erwerbstät...,laws_core
3,Art. 8 Abs. 1 AHVG,1 Vom Einkommen aus selbstständiger Erwerbstät...,laws_core
4,Art. 10 Abs. 1 AHVG,1 Nichterwerbstätige bezahlen einen Beitrag na...,laws_core


In [8]:
corpus["passage_text"] = (
    "Citation: " + corpus["citation"].astype(str).fillna("") + "\n" +
    corpus["text"].astype(str).fillna("")
).map(normalize_text)

doc_citations = corpus["citation"].astype(str).tolist()
passages = corpus["passage_text"].tolist()

print("Example passage:")
print(passages[0][:500])


Example passage:
citation: art. 1a abs. 1 ahvg 1 versichert nach diesem gesetz sind:11a.12 die natürlichen personen mit wohnsitz in der schweiz; b. die natürlichen personen, die in der schweiz eine erwerbstätigkeit ausüben; c.13 schweizer bürger, die im ausland tätig sind:1. im dienste der eidgenossenschaft,2. im dienste der internationalen organisationen, mit denen der bundesrat ein sitzabkommen abgeschlossen hat und die als arbeitgeber im sinne von artikel 12 gelten,3. im dienste privater, vom bund namhaft sub


In [9]:
# ============================================================
# Baseline6-style feature engineering layer
# - citation whitelist / blacklist
# - citation frequency + co-citation expansion
# - FR -> DE abbreviation normalization
# - regex direct citation extraction
# - train-query similarity transfer
# - law/court citation-level TF-IDF retrieval
# ============================================================

# Keep this True to match the high-scoring baseline6 idea.
# It uses the public val labels to build frequency/co-citation priors.
USE_VAL_FOR_FEATURES = True

FEATURE_TRAIN_TRANSFER_TOPK = 50
FEATURE_TRAIN_TRANSFER_TOPCIT = 80
FEATURE_LAW_DOC_TOPK = 50
FEATURE_COURT_DOC_TOPK = 50

# Use full laws_de in the TF-IDF feature branch. If this gets noisy on LB,
# set to False while keeping query-aware exact/sibling law expansion enabled.
FEATURE_USE_FULL_LAWS = True
FEATURE_COEXPAND_TOP = 30
FEATURE_MAX_CHARS_PER_CIT = 3000


def normalize_feature_text(x):
    if pd.isna(x):
        return ""
    x = str(x)
    x = re.sub(r"\s+", " ", x)
    return x.strip()


def clean_prediction_text(x: str) -> str:
    if pd.isna(x):
        return ""
    x = str(x).strip()
    x = re.sub(r"\s*;\s*", ";", x)
    x = re.sub(r";{2,}", ";", x)
    return x.strip("; ")


def normalize_query_id(x: str) -> str:
    x = str(x).strip()
    m = re.match(r"^(test|train|val)_(\d+)$", x)
    if m:
        prefix, num = m.groups()
        return f"{prefix}_{int(num):03d}"
    return x


def normalize_scores(items):
    if not items:
        return []
    vals = np.array([float(s) for _, s in items], dtype=np.float32)
    vals = np.nan_to_num(vals, nan=0.0, posinf=0.0, neginf=0.0)
    mn, mx = float(vals.min()), float(vals.max())
    if mx <= mn:
        return [(c, 1.0) for c, _ in items]
    return [(c, (float(s) - mn) / (mx - mn)) for c, s in items]


# Available prediction space:
#   - BGE dense corpus remains gold-core for speed.
#   - Full laws_de citations are allowed as query-aware expansion candidates.
#   - Court stays conservative: filtered court + direct case-style citations only.
corpus_cit_set = set(corpus["citation"].dropna().astype(str).str.strip().tolist())
law_cit_set = set(laws["citation"].dropna().astype(str).str.strip().tolist())
court_cit_set = set(court_full["citation"].dropna().astype(str).str.strip().tolist())

law_all_cit_set = set(laws_all["citation"].dropna().astype(str).str.strip().tolist())
court_all_cit_set = set(court_all["citation"].dropna().astype(str).str.strip().tolist())
court_case_cit_set = {
    c for c in court_all_cit_set
    if re.match(r"^(BGE\s+\d+\s+[IVX]+\s+\d+|[1-9][A-Z]{1,3}_\d+/\d{4}\s+E\.)", str(c))
}

corpus_set = corpus_cit_set | law_cit_set | court_cit_set | law_all_cit_set | court_case_cit_set

# baseline6 filtered these out because they were noisy/non-target citations.
BLACKLIST = {c for c in corpus_set if "KGTG" in str(c)}
corpus_set = {c for c in corpus_set if c not in BLACKLIST}

print("Feature corpus citations:", f"{len(corpus_set):,}")
print("  BGE core citation count:", f"{len(corpus_cit_set):,}")
print("  full law expansion count:", f"{len(law_all_cit_set):,}")
print("  direct court case expansion count:", f"{len(court_case_cit_set):,}")
print("Feature blacklist:", f"{len(BLACKLIST):,}")


# ------------------------
# Frequency / co-citation
# ------------------------
global_freq = Counter()
for cits in train["gold_citations"].fillna("").apply(split_citations):
    global_freq.update([c for c in cits if c in corpus_set and c not in BLACKLIST])

if USE_VAL_FOR_FEATURES:
    for cits in val["gold_citations"].fillna("").apply(split_citations):
        global_freq.update([c for c in cits if c in corpus_set and c not in BLACKLIST])

global_score = dict(global_freq)

cit_cooccur = defaultdict(Counter)
cooc_source = list(train["gold_citations"].fillna("").apply(split_citations))
if USE_VAL_FOR_FEATURES:
    cooc_source += list(val["gold_citations"].fillna("").apply(split_citations))

for cit_list in cooc_source:
    clean = [c for c in cit_list if c in corpus_set and c not in BLACKLIST]
    s = set(clean)
    for c in clean:
        for o in s:
            if o != c:
                cit_cooccur[c][o] += 1


def coexpand(seeds, top=FEATURE_COEXPAND_TOP):
    scores = Counter()
    seed_set = set(seeds)
    for c in seeds:
        w = 1.0 + global_score.get(c, 0) * 0.05
        for o, cnt in cit_cooccur.get(c, {}).items():
            if o not in BLACKLIST and o in corpus_set:
                scores[o] += cnt * w
    for c in seed_set:
        scores.pop(c, None)
    return list(seed_set) + [c for c, _ in scores.most_common(top)]


# ------------------------
# Abbreviation handling
# ------------------------
FR_TO_DE = {
    "LAI": "IVG", "LPGA": "ATSG", "LAA": "UVG", "LAMal": "KVG", "LAVS": "AHVG",
    "LPP": "BVG", "LACI": "AVIG", "LTF": "BGG", "CPC": "ZPO", "CPP": "StPO",
    "CP": "StGB", "CC": "ZGB", "CO": "OR", "LP": "SchKG", "LDIP": "IPRG",
    "LDA": "URG", "LCD": "UWG", "LPM": "MSchG", "LIFD": "DBG", "LIVA": "MWSTG",
    "LFINMA": "FINMAG", "LBA": "BankG", "LSA": "VAG", "LRFP": "PrHG", "LEI": "AIG",
    "LAsi": "AsylG", "PA": "VwVG", "LCR": "SVG", "OJ": "BGG", "PCF": "ZPO",
    "LDFR": "BGBB", "LPN": "NHG", "LAT": "RPG", "LArm": "ArG",
}

abbrev_set = set()
for cit in corpus_set:
    cit = str(cit)
    if cit.startswith("Art."):
        abbrev_set.add(cit.strip().split()[-1])

abbrev_to_cits = defaultdict(list)
for cit in corpus_set:
    cit = str(cit)
    if cit.startswith("Art.") and cit not in BLACKLIST:
        abbrev_to_cits[cit.strip().split()[-1]].append(cit)

for abbrev in abbrev_to_cits:
    abbrev_to_cits[abbrev].sort(key=lambda c: global_score.get(c, 0), reverse=True)


def normalize_abbrev(raw):
    return FR_TO_DE.get(raw, raw)


# ------------------------
# Regex extraction
# ------------------------
ART_FULL = re.compile(
    r"\b[Aa]rt(?:icle|ikel|\.)\s*\.?\s*([\d]+[a-z]*)"
    r"(?:\s+(?:Abs(?:atz|\.)?|al\.?|para\.?)\s*([\d]+[a-z]*))?"
    r"(?:\s+(?:lit\.?|let\.?)\s*[a-z])?"
    r"\s+([A-ZÄÖÜ][A-Za-z0-9äöüÄÖÜ]{1,15})\b"
)

ART_BARE = re.compile(
    r"\b[Aa]rt(?:icle|ikel|\.)\s*\.?\s*([\d]+[a-z]*)"
    r"(?:\s+(?:Abs(?:atz|\.)?|al\.?|para\.?)\s*([\d]+[a-z]*))?"
    r"(?:\s+(?:lit\.?|let\.?)\s*[a-z])?\b"
)

BGE_PAT = re.compile(r"\bBGE\s+(\d+\s+[IVX]+\s+\d+)\s+E\.\s*([\d\.a-z]+)")
CASE_PAT = re.compile(r"\b([1-9][A-Z]{1,3}_\d+/\d{4})\s+E\.\s*([\d\.a-z]+)")

ABBR_PAT = re.compile(
    r"\b("
    r"StPO|ZGB|OR|StGB|BV|ZPO|BGG|SchKG|IVG|ATSG|UVG|KVG|AHVG|BVG|AVIG|SVG|"
    r"USG|DBG|MWSTG|VVG|URG|DSG|IPRG|RPG|NHG|GwG|FusG|PatG|MSchG|ArG|"
    r"FINMAG|BankG|HMG|BetmG|AsylG|AIG|VwVG|FIDLEG|"
    r"JStG|JStPO|GBV|BGFA|PrHG|VAG|EIMP|RAG|UWG|BEHG|KAG|FinfraG|FINIG|BEG|"
    r"VStG|VZAE|EOG|FamZG|DesG|MSchV|AHVV|UVV|HVUV|"
    r"LAI|LPGA|LAA|LAMal|LAVS|LPP|LACI|LTF|CPC|CPP|CP(?!\w)|CC(?!\w)|CO(?!\w)|"
    r"LP(?!\w)|LDIP|LDA|LCD|LPM|LIFD|LIVA|LFINMA|LBA|LSA|LRFP|LEI|LAsi|LCR|PA(?!\w)"
    r")\b"
)


def try_canon(art, abs_, abbrev):
    abbrev = normalize_abbrev(abbrev)
    if abbrev not in abbrev_set:
        return None
    cands = []
    if abs_:
        cands.append(f"Art. {art} Abs. {abs_} {abbrev}")
    cands.append(f"Art. {art} {abbrev}")
    for c in cands:
        if c in corpus_set and c not in BLACKLIST:
            return c
    return None


def extract_regex_advanced(query):
    text = str(query)
    found = []
    found_set = set()

    def add(c):
        if c and c in corpus_set and c not in BLACKLIST and c not in found_set:
            found.append(c)
            found_set.add(c)

    for m in ART_FULL.finditer(text):
        add(try_canon(m.group(1), m.group(2) or "", m.group(3)))

    abbr_positions = [
        (m.start(), normalize_abbrev(m.group(1)))
        for m in ABBR_PAT.finditer(text)
        if normalize_abbrev(m.group(1)) in abbrev_set
    ]

    for m in ART_BARE.finditer(text):
        art = m.group(1)
        abs_ = m.group(2) or ""
        pos = m.start()
        best_abbrev = None
        best_dist = 999999
        for apos, abbrev in abbr_positions:
            dist = abs(apos - pos)
            if dist < best_dist and dist < 150:
                best_dist = dist
                best_abbrev = abbrev
        if best_abbrev:
            add(try_canon(art, abs_, best_abbrev))

    for m in BGE_PAT.finditer(text):
        c = f"BGE {m.group(1).strip()} E. {m.group(2).strip()}"
        if c in corpus_set and c not in BLACKLIST:
            add(c)

    for m in CASE_PAT.finditer(text):
        c = f"{m.group(1)} E. {m.group(2).strip()}"
        if c in corpus_set and c not in BLACKLIST:
            add(c)

    return found


def extract_abbrevs(query):
    raw = set(ABBR_PAT.findall(str(query)))
    result = set()
    for r in raw:
        g = normalize_abbrev(r)
        if g in abbrev_set and g not in BLACKLIST:
            result.add(g)
    return list(result)


DOMAIN_KW = {
    "testament": ["ZGB", "IPRG"], "testator": ["ZGB", "IPRG"], "holograph": ["ZGB"],
    "heir": ["ZGB", "IPRG", "SchKG"], "inherit": ["ZGB", "IPRG"], "estate": ["ZGB", "IPRG", "SchKG"],
    "custody": ["ZGB", "ZPO", "IPRG"], "divorce": ["ZGB", "ZPO", "IPRG"],
    "invalidity": ["IVG", "ATSG", "BGG"], "disability": ["IVG", "ATSG", "BGG"],
    "accident": ["UVG", "ATSG", "SVG"], "criminal": ["StGB", "StPO", "BGG"],
    "detention": ["StPO", "BGG"], "contract": ["OR", "ZPO"], "lease": ["OR", "ZPO"],
    "employment": ["OR", "ArG"], "appeal": ["BGG", "ZPO", "StPO"],
    "jurisdiction": ["IPRG", "ZPO", "BGG"], "evidence": ["ZPO", "StPO"],
    "tax": ["DBG", "MWSTG", "VStG"], "asylum": ["AsylG", "AIG", "VwVG"],
    "medical": ["UVG", "IVG", "ATSG", "OR"],
}


def get_domain_abbrevs(query):
    q = str(query).lower()
    found = set()
    for kw, abbrevs in DOMAIN_KW.items():
        if kw in q:
            found.update(abbrevs)
    return [a for a in found if a in abbrev_set and a not in BLACKLIST]


def abbrev_expand(direct_hits, abbrevs, top_per=20):
    results = []
    direct_set = set(direct_hits)
    for abbrev in abbrevs:
        cits = [c for c in abbrev_to_cits.get(abbrev, []) if c not in direct_set and c not in BLACKLIST]
        scored = []
        for cit in cits:
            co = sum(cit_cooccur.get(d, {}).get(cit, 0) for d in direct_hits)
            g = global_score.get(cit, 0)
            scored.append((cit, co * 5 + g))
        scored.sort(key=lambda x: x[1], reverse=True)
        results.extend(c for c, _ in scored[:top_per])
    return results


# ------------------------
# Query-to-query TF-IDF transfer
# ------------------------
print("Building baseline6-style query TF-IDF transfer features...")
train_qs = train["query"].astype(str).tolist()
val_qs = val["query"].astype(str).tolist()
test_qs = test["query"].astype(str).tolist()
all_qs = train_qs + val_qs + test_qs

tfidf_w = TfidfVectorizer(max_features=80000, ngram_range=(1, 2), sublinear_tf=True, strip_accents="unicode", min_df=1)
tfidf_w.fit(all_qs)
train_wv = tfidf_w.transform(train_qs)

tfidf_c = TfidfVectorizer(max_features=60000, ngram_range=(3, 5), sublinear_tf=True, analyzer="char_wb", min_df=2)
tfidf_c.fit(all_qs)
train_cv = tfidf_c.transform(train_qs)


def train_transfer_from_query(query, top_k=FEATURE_TRAIN_TRANSFER_TOPK, top_cit=FEATURE_TRAIN_TRANSFER_TOPCIT):
    qvec_w = tfidf_w.transform([str(query)])
    qvec_c = tfidf_c.transform([str(query)])
    sw = cosine_similarity(qvec_w, train_wv).flatten()
    sc = cosine_similarity(qvec_c, train_cv).flatten()
    s = 0.5 * sw + 0.5 * sc
    idx = np.argsort(s)[::-1][:top_k]
    scores = Counter()
    for i in idx:
        sim = float(s[i])
        if sim < 0.02:
            continue
        for cit in split_citations(train["gold_citations"].iloc[i]):
            if cit in corpus_set and cit not in BLACKLIST:
                scores[cit] += sim * sim
    return [(c, sc2) for c, sc2 in scores.most_common(top_cit)]


# ------------------------
# Citation-level document TF-IDF features
# ------------------------
print("Building baseline6-style law/court citation TF-IDF features...")
# Use full laws_de for TF-IDF law expansion. This is much cheaper than full dense encoding
# and helps recover legal articles that were absent from train/val gold citations.
_feature_law_source_df = laws_all if FEATURE_USE_FULL_LAWS else laws
feature_law_df = _feature_law_source_df[["citation", "text"]].copy()
feature_law_df["citation"] = feature_law_df["citation"].astype(str).str.strip()
feature_law_df["doc_text"] = feature_law_df["text"].fillna("").astype(str).str.slice(0, FEATURE_MAX_CHARS_PER_CIT)
feature_law_df = feature_law_df[~feature_law_df["citation"].isin(BLACKLIST)].reset_index(drop=True)

feature_court_df = court_full[["citation", "text"]].copy()
feature_court_df["citation"] = feature_court_df["citation"].astype(str).str.strip()
feature_court_df["doc_text"] = feature_court_df["text"].fillna("").astype(str).str.slice(0, FEATURE_MAX_CHARS_PER_CIT)
feature_court_df = feature_court_df[~feature_court_df["citation"].isin(BLACKLIST)].reset_index(drop=True)


def build_doc_tfidf(df, max_features=120000):
    docs = (df["citation"].astype(str) + " " + df["doc_text"].astype(str)).tolist()
    if len(docs) == 0 or not any(str(x).strip() for x in docs):
        return None, None
    vec = TfidfVectorizer(max_features=max_features, ngram_range=(1, 2), sublinear_tf=True, strip_accents="unicode", min_df=1)
    mat = vec.fit_transform(docs)
    return vec, mat


law_doc_vec, law_doc_mat = build_doc_tfidf(feature_law_df)
court_doc_vec, court_doc_mat = build_doc_tfidf(feature_court_df)

print("feature law TF-IDF:", None if law_doc_mat is None else law_doc_mat.shape)
print("feature court TF-IDF:", None if court_doc_mat is None else court_doc_mat.shape)

gc.collect()


def retrieve_from_feature_docs(query, vec, mat, df, topk=40):
    if vec is None or mat is None or len(df) == 0:
        return []
    qv = vec.transform([str(query)])
    sims = cosine_similarity(qv, mat).flatten()
    idx = np.argsort(sims)[::-1][:topk]
    results = []
    for i in idx:
        score = float(sims[i])
        if score <= 0:
            continue
        cit = str(df.iloc[i]["citation"])
        if cit in corpus_set and cit not in BLACKLIST:
            results.append((cit, score))
    return results


def retrieve_feature_scores(query, topn=100):
    """Return [(citation, normalized_feature_score)] from baseline6-style feature engineering."""
    scored = {}

    def add(items, weight):
        for item in items:
            c = item[0] if isinstance(item, tuple) else item
            s = float(item[1]) if isinstance(item, tuple) else 1.0
            if c not in corpus_set or c in BLACKLIST:
                continue
            scored[c] = scored.get(c, 0.0) + weight * s

    direct = extract_regex_advanced(query)
    add([(c, 1.0) for c in direct], weight=10.0)

    abbrevs = extract_abbrevs(query)
    if abbrevs:
        exp = abbrev_expand(direct, abbrevs, top_per=25)
        add([(c, 1.0) for c in exp], weight=3.0)

    domain_ab = get_domain_abbrevs(query)
    extra_ab = list(set(domain_ab) - set(abbrevs))
    if extra_ab:
        dom_exp = abbrev_expand(direct + list(scored)[:5], extra_ab, top_per=15)
        add([(c, 1.0) for c in dom_exp], weight=1.5)

    add(train_transfer_from_query(query), weight=1.5)
    add(retrieve_from_feature_docs(query, law_doc_vec, law_doc_mat, feature_law_df, topk=FEATURE_LAW_DOC_TOPK), weight=3.5)
    add(retrieve_from_feature_docs(query, court_doc_vec, court_doc_mat, feature_court_df, topk=FEATURE_COURT_DOC_TOPK), weight=2.8)

    for c in list(scored):
        scored[c] += 0.03 * global_score.get(c, 0.0)

    top_seeds = sorted(scored, key=scored.get, reverse=True)[:15]
    for c in coexpand(top_seeds, top=FEATURE_COEXPAND_TOP):
        if c not in scored:
            scored[c] = 0.005 * global_score.get(c, 0.001)

    ranked = sorted(scored.items(), key=lambda x: x[1], reverse=True)[:topn]
    return normalize_scores(ranked)


Feature corpus citations: 1,481,770
  BGE core citation count: 2,111
  full law expansion count: 175,933
  direct court case expansion count: 1,305,912
Feature blacklist: 75
Building baseline6-style query TF-IDF transfer features...
Building baseline6-style law/court citation TF-IDF features...
feature law TF-IDF: (175858, 120000)
feature court TF-IDF: (153, 24856)


In [10]:
# ============================================================
# Citation rule boost layer
# ------------------------------------------------------------
# Purpose:
#   BGE-M3 is strong semantically, but can blur legal identifiers such as
#   Art. 221 vs Art. 222, Abs. 1 vs Abs. 2, or StPO vs StGB.
#   This layer gives explicit citation-pattern matches a small hard boost
#   during final ranking. It is query-side only and does not add memory pressure.
# ============================================================

CITATION_DIRECT_BOOST = 1.25          # exact regex-canonicalized hit, e.g. Art. 221 Abs. 1 StPO
CITATION_ART_ABS_LAW_BOOST = 0.85     # same article + paragraph + law code
CITATION_ART_LAW_BOOST = 0.55         # same article + law code
CITATION_LAW_ONLY_BOOST = 0.05        # weak prior for law-code/domain matches
CITATION_MAX_BOOST_CANDIDATES = 120   # safety cap per query


def _citation_parts(cit):
    """Parse canonical law citations, including optional Abs. and lit. parts."""
    cit = str(cit).strip()
    m = re.match(
        r"^Art\.\s*([0-9]+[a-zA-Z]*)"
        r"\s*(?:Abs\.\s*([0-9]+[a-zA-Z]*))?"
        r"\s*(?:lit\.\s*([a-zA-Z]))?"
        r"\s+([A-Za-zÄÖÜäöü0-9_.-]+)$",
        cit,
    )
    if not m:
        return None
    return {
        "art": str(m.group(1)).strip(),
        "abs": str(m.group(2) or "").strip(),
        "lit": str(m.group(3) or "").strip().lower(),
        "law": normalize_abbrev(m.group(4)),
    }


def _query_citation_specs(query):
    """Extract structured citation specs from the query using the existing regex objects."""
    text = str(query)
    specs = []
    seen = set()

    def add(art, abs_, law):
        law = normalize_abbrev(law)
        if not art or law not in abbrev_set:
            return
        key = (str(art).strip(), str(abs_ or "").strip(), law)
        if key not in seen:
            seen.add(key)
            specs.append({"art": key[0], "abs": key[1], "law": key[2]})

    # Full pattern: Art. 221 Abs. 1 StPO / Art. 221 StPO
    for m in ART_FULL.finditer(text):
        add(m.group(1), m.group(2) or "", m.group(3))

    # Bare Art. pattern + nearest legal abbreviation in the query.
    abbr_positions = [
        (m.start(), normalize_abbrev(m.group(1)))
        for m in ABBR_PAT.finditer(text)
        if normalize_abbrev(m.group(1)) in abbrev_set
    ]
    for m in ART_BARE.finditer(text):
        art = m.group(1)
        abs_ = m.group(2) or ""
        pos = m.start()
        if not abbr_positions:
            continue
        apos, law = min(abbr_positions, key=lambda x: abs(x[0] - pos))
        if abs(apos - pos) < 180:
            add(art, abs_, law)

    return specs




# ------------------------------------------------------------
# Robust debug/helper extractor + soft candidate-level fallback
# ------------------------------------------------------------
# The strict citation boost above only fires when the query contains a fairly
# standard citation pattern. Many competition queries mention article numbers
# or law abbreviations in looser forms, so we also expose extract_citation()
# for debugging and apply a small numeric/law-code boost to candidates that
# are already retrieved by BGE/features. This changes ranking without adding
# large noisy candidate pools.

CITATION_NUMERIC_ART_BOOST = 0.12      # query number equals candidate article number
CITATION_NUMERIC_ABS_BOOST = 0.04      # query number equals candidate Abs. number
CITATION_SOFT_LAW_BOOST = 0.05         # query law abbreviation equals candidate law code
CITATION_SOFT_MAX_BOOST = 0.25         # cap soft boost per candidate

def is_valid_article_num(x):
    x = str(x).strip()
    digits = re.sub(r"\D", "", x)

    if not digits:
        return False

    n = int(digits)

    # 过滤年份或疑似年份残片
    if 180 <= n <= 2099:
        return False

    # 过滤明显日期/月份小数字，除非是 Abs. 这种你已经明确抽到的
    if n > 1000:
        return False

    return True

def extract_citation(text):
    """
    Citation-aware parser for debugging and soft boosting.
    Only extracts legally meaningful article numbers and known law abbreviations.
    """
    text = str(text)

    laws = set()
    for m in re.finditer(r"\b[A-Za-zÄÖÜäöü]{2,10}\b", text):
        abbr = normalize_abbrev(m.group(0))
        if abbr in abbrev_set:
            laws.add(abbr)

    numbers = set()

    # 1) Art. 221 / Article 221 这种明确法条编号
    for m in re.finditer(r"\b(?:Art\.?|Article)\s*(\d+[a-zA-Z]?)\b", text, flags=re.I):
        numbers.add(m.group(1))

    # 2) 221 StPO / 17 IVG 这种 law code 附近编号
    law_pattern = "|".join(map(re.escape, sorted(abbrev_set, key=len, reverse=True)))
    for m in re.finditer(rf"\b(\d+[a-zA-Z]?)\s*(?:{law_pattern})\b", text):
        numbers.add(m.group(1))

    # 3) 保留 citation 里 Abs. 后面的编号，但只在出现 Art. 时启用
    if re.search(r"\bArt\.?\b", text, flags=re.I):
        for m in re.finditer(r"\bAbs\.?\s*(\d+[a-zA-Z]?)\b", text, flags=re.I):
            numbers.add(m.group(1))

    numbers = {x for x in numbers if is_valid_article_num(x)}
    
    return {"numbers": numbers, "laws": laws}


def citation_candidate_soft_boost(query, candidate_citation):
    """
    Small, safe fallback boost for candidates that are already in score_map.
    It does NOT introduce new candidates; it only reorders existing candidates.
    """
    q = extract_citation(query)
    if not q["numbers"] and not q["laws"]:
        return 0.0

    parts = _citation_parts(candidate_citation)
    if not parts:
        return 0.0

    boost = 0.0

    cand_law = parts.get("law", "")
    cand_art = str(parts.get("art", ""))
    cand_abs = str(parts.get("abs", ""))

    if cand_law and cand_law in q["laws"]:
        boost += CITATION_SOFT_LAW_BOOST

    if cand_art and cand_art in q["numbers"]:
        boost += CITATION_NUMERIC_ART_BOOST

    if cand_abs and cand_abs in q["numbers"]:
        boost += CITATION_NUMERIC_ABS_BOOST

    return min(boost, CITATION_SOFT_MAX_BOOST)


def citation_rule_boost_scores(query, max_candidates=CITATION_MAX_BOOST_CANDIDATES):
    """
    Return [(citation, boost_score)] based on exact and partial legal citation matches.
    This intentionally only touches a small set of candidates derived from the query's law codes.
    """
    boosts = Counter()

    # 1) Exact canonical citation hits from the existing advanced extractor.
    direct_hits = extract_regex_advanced(query)
    for cit in direct_hits:
        if cit in corpus_set and cit not in BLACKLIST:
            boosts[cit] += CITATION_DIRECT_BOOST

    # 2) Partial structured boost for article/law and article/paragraph/law matches.
    specs = _query_citation_specs(query)
    for spec in specs:
        law = spec["law"]
        art = spec["art"]
        abs_ = spec["abs"]
        checked = 0
        for cit in abbrev_to_cits.get(law, []):
            if checked >= max_candidates:
                break
            if cit in BLACKLIST or cit not in corpus_set:
                continue
            parts = _citation_parts(cit)
            if not parts:
                continue
            checked += 1
            if parts["law"] != law or parts["art"] != art:
                continue
            if abs_ and parts["abs"] == abs_:
                boosts[cit] += CITATION_ART_ABS_LAW_BOOST
            else:
                boosts[cit] += CITATION_ART_LAW_BOOST

    # 3) Very weak law-code/domain prior. This should not dominate ranking.
    for law in get_domain_abbrevs(query):
        added = 0
        for cit in abbrev_to_cits.get(law, []):
            if cit in corpus_set and cit not in BLACKLIST:
                boosts[cit] += CITATION_LAW_ONLY_BOOST
                added += 1
            if added >= 20:
                break

    return sorted(boosts.items(), key=lambda x: x[1], reverse=True)



In [11]:
# ============================================================
# Query-aware law expansion
# ------------------------------------------------------------
# Dense BGE still uses the small gold-core corpus, but this layer dynamically
# adds relevant laws_de citations that were filtered out by train/val gold.
# It is intentionally law-heavy and court-conservative.
# ============================================================

LAW_EXPANSION_ENABLED = True

# Direct article-law hits should be competitive, sibling articles should be weak.
LAW_EXPANSION_EXACT_SCORE = 1.05
LAW_EXPANSION_SAME_ART_SCORE = 0.78
LAW_EXPANSION_SIBLING_SCORE = 0.22
LAW_EXPANSION_DOMAIN_PRIOR_SCORE = 0.025

LAW_EXPANSION_SIBLING_WINDOW = 3
LAW_EXPANSION_MAX_ITEMS = 120
LAW_EXPANSION_DOMAIN_PRIOR_TOPN = 25

# Optional: exact case citations from full court, but no broad court semantic expansion.
COURT_DIRECT_CASE_EXPANSION_SCORE = 0.75


def _build_law_expansion_indices():
    exact = defaultdict(list)      # (law, art) -> citations
    by_law = defaultdict(list)     # law -> citations
    all_parts = {}

    for cit in laws_all["citation"].dropna().astype(str).str.strip().unique().tolist():
        if cit in BLACKLIST:
            continue
        parts = _citation_parts(cit)
        if not parts:
            continue
        law = parts["law"]
        art = parts["art"]
        exact[(law, art)].append(cit)
        by_law[law].append(cit)
        all_parts[cit] = parts

    # Put train/val-frequent citations first, then article number order when possible.
    def sort_key(c):
        p = all_parts.get(c, {})
        art = str(p.get("art", ""))
        art_num = int(re.sub(r"\D", "", art) or 10**9)
        return (-global_score.get(c, 0.0), art_num, c)

    for k in exact:
        exact[k] = sorted(set(exact[k]), key=sort_key)
    for k in by_law:
        by_law[k] = sorted(set(by_law[k]), key=sort_key)

    return exact, by_law, all_parts


law_expansion_exact, law_expansion_by_law, law_expansion_parts = _build_law_expansion_indices()

court_direct_case_set = {
    c for c in court_all["citation"].dropna().astype(str).str.strip().unique().tolist()
    if c not in BLACKLIST and c in corpus_set and (
        re.match(r"^[1-9][A-Z]{1,3}_\d+/\d{4}\s+E\.", c) or c.startswith("BGE ")
    )
}

print("law_expansion_exact keys:", len(law_expansion_exact))
print("law_expansion_by_law keys:", len(law_expansion_by_law))
print("court_direct_case_set:", len(court_direct_case_set))


def _article_to_int(art):
    m = re.match(r"^(\d+)", str(art))
    return int(m.group(1)) if m else None


def query_aware_law_expansion_scores(query, max_items=LAW_EXPANSION_MAX_ITEMS):
    """
    Return [(citation, score)] for laws_de citations outside/inside the BGE core.
    This focuses on:
      1) exact query article + law code
      2) same article with different Abs.
      3) nearby sibling articles within the same law code
      4) very weak domain prior for law codes detected from the query
    """
    if not LAW_EXPANSION_ENABLED:
        return []

    boosts = Counter()
    specs = _query_citation_specs(query)

    for spec in specs:
        law = spec["law"]
        art = spec["art"]
        abs_ = spec["abs"]
        art_int = _article_to_int(art)

        # Exact article/law expansion: all Abs. under this article are allowed.
        for cit in law_expansion_exact.get((law, art), []):
            parts = law_expansion_parts.get(cit) or {}
            if abs_ and parts.get("abs") == abs_:
                boosts[cit] += LAW_EXPANSION_EXACT_SCORE
            else:
                boosts[cit] += LAW_EXPANSION_SAME_ART_SCORE

        # Sibling article expansion: weak, only within a small window.
        if art_int is not None:
            for delta in range(-LAW_EXPANSION_SIBLING_WINDOW, LAW_EXPANSION_SIBLING_WINDOW + 1):
                if delta == 0:
                    continue
                sib_art = str(art_int + delta)
                if int(sib_art) <= 0:
                    continue
                sibling_weight = LAW_EXPANSION_SIBLING_SCORE / (abs(delta) ** 0.7)
                for cit in law_expansion_exact.get((law, sib_art), [])[:8]:
                    boosts[cit] += sibling_weight

    # Very weak prior for query law code. This fills candidates when query states a law
    # but no article is parsed. It should not dominate exact/semantic signals.
    for law in get_domain_abbrevs(query):
        added = 0
        for cit in law_expansion_by_law.get(law, []):
            if cit in boosts:
                continue
            boosts[cit] += LAW_EXPANSION_DOMAIN_PRIOR_SCORE
            added += 1
            if added >= LAW_EXPANSION_DOMAIN_PRIOR_TOPN:
                break

    return sorted(boosts.items(), key=lambda x: (x[1], global_score.get(x[0], 0.0)), reverse=True)[:max_items]


def query_aware_court_direct_scores(query):
    """
    Conservative court expansion: only exact case citations already mentioned by the query.
    No full-court semantic search here because court_considerations is noisy and large.
    """
    hits = []
    for cit in extract_regex_advanced(query):
        if cit in court_direct_case_set and cit not in BLACKLIST:
            hits.append((cit, COURT_DIRECT_CASE_EXPANSION_SCORE))
    return hits


law_expansion_exact keys: 70107
law_expansion_by_law keys: 2034
court_direct_case_set: 1305912


In [12]:
# ============================================================
# Conservative citation-only rescue layer (v4-safe)
# ------------------------------------------------------------
# This is the safe version of the teacher/paper-inspired idea:
#   - Do NOT add broad legal reasoning words to the dense query.
#   - Do NOT run multi-query dense retrieval.
#   - Only use explicit citation signals already present in the query:
#       Art. number, Abs., law code, exact case ids.
#   - Use it as a small rescue/tie-break layer, not as the main ranking signal.
# ============================================================

SAFE_RESCUE_EXACT_ART_ABS_SCORE = 1.00
SAFE_RESCUE_SAME_ART_SCORE = 0.55
SAFE_RESCUE_DIRECT_CASE_SCORE = 0.80
SAFE_RESCUE_MAX_ITEMS = 40

# Keep siblings OFF by default. Nearby articles can help recall, but they are also
# the easiest way to pollute exact-citation F1 on the leaderboard.
SAFE_RESCUE_ENABLE_SIBLINGS = False
SAFE_RESCUE_SIBLING_SCORE = 0.08
SAFE_RESCUE_SIBLING_WINDOW = 1

# Tiny verifier scores. These only reorder candidates already in score_map.
SAFE_VERIFIER_LAW_SCORE = 0.06
SAFE_VERIFIER_ART_SCORE = 0.12
SAFE_VERIFIER_ABS_SCORE = 0.04
SAFE_VERIFIER_DIRECT_TEXT_SCORE = 0.18
SAFE_VERIFIER_MAX_SCORE = 0.30


def conservative_citation_rescue_scores(query, max_items=SAFE_RESCUE_MAX_ITEMS):
    """
    Return a small set of citation-only rescue candidates.

    Important difference from broad query expansion:
      - No generic legal concepts are appended.
      - No domain prior is added.
      - No semantic court expansion is used.
      - It only fires when the query contains explicit citation-like evidence.
    """
    boosts = Counter()
    specs = _query_citation_specs(query)

    # 1) Exact canonical citations/case citations directly mentioned in the query.
    for cit in extract_regex_advanced(query):
        cit = str(cit).strip()
        if cit in BLACKLIST:
            continue
        if cit in corpus_set or cit in law_expansion_parts or cit in court_direct_case_set:
            boosts[cit] += SAFE_RESCUE_DIRECT_CASE_SCORE

    # 2) Article + law code rescue from laws_de.
    #    This is citation-only and much more conservative than broad expansion.
    for spec in specs:
        law = spec["law"]
        art = spec["art"]
        abs_ = spec["abs"]
        art_int = _article_to_int(art)

        for cit in law_expansion_exact.get((law, art), []):
            if cit in BLACKLIST:
                continue
            parts = law_expansion_parts.get(cit) or {}
            if abs_ and parts.get("abs") == abs_:
                boosts[cit] += SAFE_RESCUE_EXACT_ART_ABS_SCORE
            else:
                boosts[cit] += SAFE_RESCUE_SAME_ART_SCORE

        # OFF by default: tiny nearby article rescue only if explicitly enabled.
        if SAFE_RESCUE_ENABLE_SIBLINGS and art_int is not None:
            for delta in range(-SAFE_RESCUE_SIBLING_WINDOW, SAFE_RESCUE_SIBLING_WINDOW + 1):
                if delta == 0:
                    continue
                sib_art = str(art_int + delta)
                if int(sib_art) <= 0:
                    continue
                for cit in law_expansion_exact.get((law, sib_art), [])[:4]:
                    if cit not in BLACKLIST:
                        boosts[cit] += SAFE_RESCUE_SIBLING_SCORE / abs(delta)

    return sorted(
        boosts.items(),
        key=lambda x: (x[1], global_score.get(x[0], 0.0)),
        reverse=True
    )[:max_items]


def safe_citation_verifier_score(query, candidate_citation):
    """
    Very small candidate-level verifier.
    It should act like a tie-breaker, not a reranker.
    """
    query = str(query)
    cit = str(candidate_citation).strip()
    q = extract_citation(query)

    if not q["numbers"] and not q["laws"]:
        return 0.0

    parts = _citation_parts(cit)
    if not parts:
        # Case citations / BGE direct text match only.
        return SAFE_VERIFIER_DIRECT_TEXT_SCORE if cit and cit.lower() in query.lower() else 0.0

    score = 0.0
    cand_law = parts.get("law", "")
    cand_art = str(parts.get("art", ""))
    cand_abs = str(parts.get("abs", ""))

    if cand_law and cand_law in q["laws"]:
        score += SAFE_VERIFIER_LAW_SCORE

    if cand_art and cand_art in q["numbers"]:
        score += SAFE_VERIFIER_ART_SCORE

    if cand_abs and cand_abs in q["numbers"]:
        score += SAFE_VERIFIER_ABS_SCORE

    if cit.lower() in query.lower():
        score += SAFE_VERIFIER_DIRECT_TEXT_SCORE

    return min(score, SAFE_VERIFIER_MAX_SCORE)


# ============================================================
# v5 safe tail replacement helper
# ------------------------------------------------------------
# The v4 rescue mostly changed ordering, not the final citation set.
# v5 makes the rescue layer able to replace only the tail of the output,
# and only when explicit citation signals are present.
# ============================================================

def has_high_confidence_citation_signal(query):
    """True only for queries with explicit law+article or exact case/BGE style signals."""
    specs = _query_citation_specs(query)
    if len(specs) > 0:
        return True
    direct = extract_regex_advanced(query)
    return len(direct) > 0


def safe_tail_replace_results(
    query,
    ranked,
    top_k=25,
    keep_topn=22,
    add_topn=3,
    min_rescue_score=0.55,
):
    """
    Keep the original head unchanged, then replace only the tail with
    high-confidence citation-only rescue candidates.

    This is designed for exact-citation F1:
      - It does not reorder the top head.
      - It only changes the final set when rescue candidates are absent.
      - It only fires on explicit citation signals.
    """
    ranked = [str(c).strip() for c in ranked if str(c).strip() and str(c).strip() not in BLACKLIST]
    if not has_high_confidence_citation_signal(query):
        return ranked[:top_k]

    keep_topn = max(0, min(int(keep_topn), int(top_k)))
    add_topn = max(0, min(int(add_topn), int(top_k) - keep_topn))

    head = []
    seen = set()
    for cit in ranked:
        if cit not in seen:
            head.append(cit)
            seen.add(cit)
        if len(head) >= keep_topn:
            break

    rescue = []
    if "conservative_citation_rescue_scores" in globals():
        for cit, s in conservative_citation_rescue_scores(query, max_items=SAFE_RESCUE_MAX_ITEMS):
            cit = str(cit).strip()
            if cit in BLACKLIST or cit in seen:
                continue
            # Require enough confidence: exact/same article or direct case level.
            if float(s) >= float(min_rescue_score):
                rescue.append(cit)
                seen.add(cit)
            if len(rescue) >= add_topn:
                break

    # If no new high-confidence rescue exists, preserve original output.
    if not rescue:
        return ranked[:top_k]

    out = head + rescue

    # Fill remaining slots from the original ranking, preserving order.
    for cit in ranked:
        if cit not in seen:
            out.append(cit)
            seen.add(cit)
        if len(out) >= top_k:
            break

    return out[:top_k]


# ============================================================
# v6 enhanced safe-tail override
# ------------------------------------------------------------
# Keep the original BGE + feature ranking untouched. Only the final tail can change.
# Additions are intentionally tail-only:
#   1) Abs-aware verifier: same Art/Law + exact Abs is rewarded; wrong Abs is penalized.
#   2) Adjacent article rescue: same law and article +/- 1 can enter only if confidence is high.
#   3) Confident co-citation: expands only from the protected head, and cannot pass alone.
#   4) Query-aware law expansion is used as a rescue source, not as a head reranker.
# ============================================================

V6_TAIL_USE_LAW_EXPANSION_RESCUE = True
V6_TAIL_LAW_EXPANSION_WEIGHT = 0.75
V6_TAIL_USE_COCIT_RESCUE = True
V6_TAIL_COCIT_WEIGHT = 0.12
V6_TAIL_COCIT_TOP_SEEDS = 8
V6_TAIL_COCIT_TOP_EACH = 3
V6_TAIL_ENABLE_ADJACENT_ARTICLE = True
V6_TAIL_ADJACENT_ARTICLE_SCORE = 0.12
V6_TAIL_SAME_ART_LAW_SCORE = 0.30
V6_TAIL_EXACT_ABS_BONUS = 0.35
V6_TAIL_WRONG_ABS_PENALTY = 0.50
V6_TAIL_CONTEXT_BONUS = 0.03
V6_TAIL_REQUIRE_SIGNAL = True

V6_STRONG_CONTEXT_WORDS = {
    "entscheidend", "massgebend", "gestützt", "gemäss", "verletzt",
    "voraussetzung", "haft", "beschwerde", "zuständigkeit", "beweis",
    "recht", "rechtsverletzung", "anspruch", "entscheid", "urteils",
}


def _v6_article_int(art):
    m = re.match(r"^(\d+)", str(art))
    return int(m.group(1)) if m else None


def _v6_tail_structural_adjustment(query, cit):
    """Small verifier for candidate tail rescue scores."""
    parts = _citation_parts(cit)
    if not parts:
        # Exact case/BGE textual mentions are already handled by conservative rescue.
        return 0.0

    score = 0.0
    specs = _query_citation_specs(query)
    if not specs:
        return 0.0

    cand_art = str(parts.get("art", ""))
    cand_abs = str(parts.get("abs", ""))
    cand_law = str(parts.get("law", ""))
    cand_art_int = _v6_article_int(cand_art)

    for spec in specs:
        q_art = str(spec.get("art", ""))
        q_abs = str(spec.get("abs", ""))
        q_law = str(spec.get("law", ""))
        q_art_int = _v6_article_int(q_art)

        if q_law != cand_law:
            continue

        if q_art == cand_art:
            score += V6_TAIL_SAME_ART_LAW_SCORE
            if q_abs and cand_abs and q_abs == cand_abs:
                score += V6_TAIL_EXACT_ABS_BONUS
            elif q_abs and cand_abs and q_abs != cand_abs:
                score -= V6_TAIL_WRONG_ABS_PENALTY
            elif q_abs and not cand_abs:
                score += 0.08

        elif V6_TAIL_ENABLE_ADJACENT_ARTICLE and q_art_int is not None and cand_art_int is not None:
            if abs(q_art_int - cand_art_int) == 1:
                # Adjacent articles are useful only as tail recall, never as main ranking.
                score += V6_TAIL_ADJACENT_ARTICLE_SCORE

    # Tiny context bonus only when query contains legal-decision context words.
    qlow = str(query).lower()
    if any(w in qlow for w in V6_STRONG_CONTEXT_WORDS):
        score += V6_TAIL_CONTEXT_BONUS

    return score


def _v6_get_law_from_citation(cit):
    m = re.search(
        r"\b(ZGB|OR|StGB|StPO|ZPO|BGG|SVG|SchKG|IPRG|AIG|ATSG|KVG|UVG|AHVG|IVG|URG|MSchG|UWG|MWSTG|DBG|VStG|ArG|BVG|PrHG|BV|EMRK)\b",
        str(cit),
        flags=re.IGNORECASE,
    )
    return m.group(1).lower() if m else None


def _v6_combined_tail_rescue_scores(query, ranked_head):
    rescue = Counter()

    # query 中明确出现的 law
    query_laws = set()
    if "extract_query_citations" in globals():
        for h in extract_query_citations(query):
            law = _v6_get_law_from_citation(h)
            if law:
                query_laws.add(law)

    # 当前 protected head 里的 law，允许 law expansion 只围绕这些 law
    head_laws = set()
    for item in ranked_head[:10]:
        seed = item[0] if isinstance(item, (tuple, list)) else item
        law = _v6_get_law_from_citation(seed)
        if law:
            head_laws.add(law)

    allowed_laws = query_laws | head_laws

    def _allow_tail_cit(cit):
        if not V6_RESTRICT_TAIL_TO_QUERY_OR_HEAD_LAW:
            return True
        law = _v6_get_law_from_citation(cit)
        if law is None:
            return True   # BGE / court case 不拦
        if not allowed_laws:
            return True
        return law in allowed_laws

    # 1) Original conservative citation rescue.
    if "conservative_citation_rescue_scores" in globals():
        for cit, s in conservative_citation_rescue_scores(query, max_items=SAFE_RESCUE_MAX_ITEMS):
            cit = str(cit).strip()
            if cit and cit not in BLACKLIST and _allow_tail_cit(cit):
                rescue[cit] += float(s)

    # 2) Query-aware laws_de expansion, tail-only.
    # 改4：只允许 query/head law 范围内的 law expansion
    if V6_TAIL_USE_LAW_EXPANSION_RESCUE and "query_aware_law_expansion_scores" in globals():
        for cit, s in query_aware_law_expansion_scores(query, max_items=LAW_EXPANSION_MAX_ITEMS):
            cit = str(cit).strip()
            if cit and cit not in BLACKLIST and _allow_tail_cit(cit):
                rescue[cit] += V6_TAIL_LAW_EXPANSION_WEIGHT * float(s)

    # 3) Exact court case expansion, tail-only.
    if "query_aware_court_direct_scores" in globals():
        for cit, s in query_aware_court_direct_scores(query):
            cit = str(cit).strip()
            if cit and cit not in BLACKLIST:
                rescue[cit] += float(s)

    # 4) Confident co-citation expansion from protected head.
    if V6_TAIL_USE_COCIT_RESCUE:
        if "cit_cooccur" in globals():
            co_map = cit_cooccur
        elif "cooccur" in globals():
            co_map = cooccur
        elif "co_citation" in globals():
            co_map = co_citation
        elif "co_citation_counter" in globals():
            co_map = co_citation_counter
        else:
            co_map = {}

        for item in ranked_head[:V6_TAIL_COCIT_SEED_TOPN]:
            seed = item[0] if isinstance(item, (tuple, list)) else item
            seed = str(seed).strip()

            if seed in BLACKLIST:
                continue

            cc_dict = co_map.get(seed, {})
            cc_items = sorted(cc_dict.items(), key=lambda x: x[1], reverse=True)[:V6_TAIL_COCIT_TOP_EACH]

            for cc, cnt in cc_items:
                cc = str(cc).strip()
                if not cc or cc in BLACKLIST or not _allow_tail_cit(cc):
                    continue

                rescue[cc] += V6_TAIL_COCIT_WEIGHT * min(float(cnt) / 5.0, 1.0)

    # 5) Structural verifier / penalties + freq bias
    for cit in list(rescue.keys()):
        rescue[cit] += _v6_tail_structural_adjustment(query, cit)

        # 改3：高频 citation 微弱加分
        if "citation_freq" in globals() and cit in citation_freq:
            denom = max(citation_freq.values()) if len(citation_freq) else 1
            rescue[cit] += V6_FREQ_BIAS_WEIGHT * (citation_freq[cit] / denom)

        if rescue[cit] <= 0:
            rescue.pop(cit, None)

    return sorted(
        rescue.items(),
        key=lambda x: (x[1], global_score.get(x[0], 0.0)),
        reverse=True,
    )

def safe_tail_replace_results(
    query,
    ranked,
    top_k=25,
    keep_topn=21,
    add_topn=4,
    min_rescue_score=0.60,
):
    """
    v6 + law-family: preserve the head, repair the tail, then allow 1 law-family slot.
    """
    ranked = [str(c).strip() for c in ranked if str(c).strip() and str(c).strip() not in BLACKLIST]

    if V6_TAIL_REQUIRE_SIGNAL and not has_high_confidence_citation_signal(query):
        return ranked[:top_k]

    keep_topn = max(0, min(int(keep_topn), int(top_k)))
    add_topn = max(0, min(int(add_topn), int(top_k) - keep_topn))

    head = []
    seen = set()
    for cit in ranked:
        if cit not in seen:
            head.append(cit)
            seen.add(cit)
        if len(head) >= keep_topn:
            break

    rescue = []
    for cit, s in _v6_combined_tail_rescue_scores(query, head):
        cit = str(cit).strip()
        if cit in BLACKLIST or cit in seen:
            continue
        if float(s) >= float(min_rescue_score):
            rescue.append(cit)
            seen.add(cit)
        if len(rescue) >= add_topn:
            break

    out = head + rescue

    for cit in ranked:
        if cit not in seen:
            out.append(cit)
            seen.add(cit)
        if len(out) >= top_k:
            break

    out = out[:top_k]

    # law-family rescue: only allowed to replace the last slot
    if "law_family_tail_rescue_scores" in globals() and BEST_USE_LAW_FAMILY_RESCUE:
        candidate_pool = set(ranked) | set(out)

        # 尽量加入已有全局候选池
        if "global_score" in globals():
            candidate_pool |= set(global_score.keys())
        if "all_law_citations" in globals():
            candidate_pool |= set(all_law_citations)
        if "gold_citation_set" in globals():
            candidate_pool |= set(gold_citation_set)

        lf_scores = law_family_tail_rescue_scores(
            query=query,
            ranked_head=head,
            candidate_pool=candidate_pool,
        )

        lf_sorted = [
            c for c, s in sorted(lf_scores.items(), key=lambda x: x[1], reverse=True)
            if float(s) >= BEST_LAW_FAMILY_TAIL_MIN_SCORE
        ]

        added = 0
        for c in lf_sorted:
            c = str(c).strip()
            if not c or c in BLACKLIST or c in out:
                continue

            # 只抢最后 BEST_LAW_FAMILY_TAIL_SLOTS 个位置，默认 1
            for _ in range(BEST_LAW_FAMILY_TAIL_SLOTS):
                if added >= BEST_LAW_FAMILY_TAIL_SLOTS:
                    break
                out[-1 - added] = c
                added += 1
                break

            if added >= BEST_LAW_FAMILY_TAIL_SLOTS:
                break

    return out[:top_k]


In [13]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

model = BGEM3FlagModel(
    "BAAI/bge-m3",
    use_fp16=True,
    device=device
)

print("Loaded BAAI/bge-m3")

device: cuda


config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/444 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loaded BAAI/bge-m3


In [14]:
BATCH_SIZE = 32
CHUNK_SIZE = 2000
MAX_LENGTH = 1024

all_dense = []
doc_sparse = []

for start in tqdm(range(0, len(passages), CHUNK_SIZE), desc="Encoding corpus with BGE-M3 dense + lexical"):
    end = min(start + CHUNK_SIZE, len(passages))
    batch_passages = passages[start:end]

    outputs = model.encode(
        batch_passages,
        batch_size=BATCH_SIZE,
        max_length=MAX_LENGTH,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False
    )

    all_dense.append(outputs["dense_vecs"].astype("float32"))
    doc_sparse.extend(outputs["lexical_weights"])

    del outputs, batch_passages
    gc.collect()
    torch.cuda.empty_cache()

doc_dense = np.vstack(all_dense).astype("float32")
del all_dense
gc.collect()

print("doc_dense:", doc_dense.shape)
print("doc_sparse:", len(doc_sparse))


Encoding corpus with BGE-M3 dense + lexical:   0%|          | 0/2 [00:00<?, ?it/s]



initial target device:   0%|          | 0/2 [00:00<?, ?it/s]

initial target device:  50%|█████     | 1/2 [00:09<00:09,  9.54s/it]

initial target device: 100%|██████████| 2/2 [00:18<00:00,  9.36s/it]


Inference Embeddings:  91%|█████████ | 29/32 [00:03<00:00, 19.68it/s]

Inference Embeddings: 100%|██████████| 32/32 [00:03<00:00, 10.19it/s]


Chunks: 100%|██████████| 2/2 [00:05<00:00,  2.82s/it]


Inference Embeddings: 100%|██████████| 2/2 [00:01<00:00,  1.32it/s]


Chunks: 100%|██████████| 2/2 [00:01<00:00,  1.20it/s]


doc_dense: (2114, 1024)
doc_sparse: 2114


In [15]:
faiss.normalize_L2(doc_dense)

index = faiss.IndexFlatIP(doc_dense.shape[1])
index.add(doc_dense)

print("FAISS index size:", index.ntotal)

FAISS index size: 2114


In [16]:
def sparse_score(q_sparse, d_sparse):
    score = 0.0
    for k, v in q_sparse.items():
        if k in d_sparse:
            score += float(v) * float(d_sparse[k])
    return score


def retrieve(
    query,
    top_k=10,
    dense_topn=150,
    dense_weight=0.80,
    lexical_weight=0.20,
    sparse_weight=None,   # backward-compatible alias
):
    if sparse_weight is not None:
        lexical_weight = sparse_weight

    q = normalize_text(query)

    q_outputs = model.encode(
        [q],
        batch_size=1,
        max_length=512,
        return_dense=True,
        return_sparse=True,
        return_colbert_vecs=False
    )

    q_dense = q_outputs["dense_vecs"].astype("float32")
    q_sparse = q_outputs["lexical_weights"][0]

    faiss.normalize_L2(q_dense)

    dense_scores, dense_idx = index.search(q_dense, dense_topn)
    dense_scores = dense_scores[0]
    dense_idx = dense_idx[0]

    lexical_scores = []
    for idx in dense_idx:
        idx = int(idx)
        if idx < 0:
            lexical_scores.append(0.0)
        else:
            lexical_scores.append(sparse_score(q_sparse, doc_sparse[idx]))

    final_scores = (
        dense_weight * minmax(dense_scores)
        + lexical_weight * minmax(lexical_scores)
    )
    order = np.argsort(-final_scores)

    results = []
    seen = set()

    for j in order:
        idx = int(dense_idx[j])
        if idx < 0:
            continue
        cit = str(doc_citations[idx]).strip()
        if cit in BLACKLIST:
            continue

        if cit not in seen:
            results.append(cit)
            seen.add(cit)

        if len(results) >= top_k:
            break

    return results


In [17]:
def retrieve_final(
    query,
    top_k=10,
    bge_topk=10,
    dense_topn=150,
    dense_weight=0.80,
    lexical_weight=0.20,
    use_features=True,
    feature_topn=120,
    feature_weight=0.70,
    use_citation_boost=True,
    citation_boost_weight=1.00,
    use_law_expansion=True,
    law_expansion_weight=0.65,
    use_safe_citation_rescue=False,
    safe_rescue_weight=0.10,
    use_safe_verifier=False,
    safe_verifier_weight=0.05,
    use_safe_tail_replace=False,
    safe_tail_keep_topn=22,
    safe_tail_add_topn=3,
    safe_tail_min_score=0.55,
):
    # Pure-BGE backbone + baseline6-style feature layer + citation rule boost.
    # v4-safe additions are citation-only: no broad query expansion and no multi-query dense retrieval.
    bge_preds = retrieve(
        query,
        top_k=bge_topk,
        dense_topn=dense_topn,
        dense_weight=dense_weight,
        lexical_weight=lexical_weight,
    )

    score_map = {}

    def add_ranked(cits, weight):
        n = max(len(cits), 1)
        for r, cit in enumerate(cits):
            cit = str(cit).strip()
            if cit in BLACKLIST:
                continue
            score_map[cit] = score_map.get(cit, 0.0) + weight * (n - r) / n

    add_ranked(bge_preds, weight=1.00)

    if use_features:
        feature_hits = retrieve_feature_scores(query, topn=feature_topn)
        for cit, s in feature_hits:
            cit = str(cit).strip()
            if cit in BLACKLIST:
                continue
            score_map[cit] = score_map.get(cit, 0.0) + feature_weight * float(s)

    # Hard but bounded correction for explicit legal identifiers.
    # This is deliberately applied after BGE/features so exact Art./Abs./law matches
    # are hard to push out of top-k, while non-matching candidates remain ranked by model scores.
    if use_citation_boost:
        # 1) Strict citation-pattern boost: can add exact/partial citation hits.
        for cit, boost in citation_rule_boost_scores(query):
            cit = str(cit).strip()
            if cit in BLACKLIST:
                continue
            score_map[cit] = score_map.get(cit, 0.0) + citation_boost_weight * float(boost)

        # 2) Soft numeric/law fallback: only re-rank candidates already retrieved
        #    by BGE/features/strict boost, so it is much less noisy than scanning
        #    the whole corpus for every number in the query.
        for cit in list(score_map.keys()):
            soft = citation_candidate_soft_boost(query, cit)
            if soft > 0:
                score_map[cit] = score_map.get(cit, 0.0) + citation_boost_weight * float(soft)

    # Query-aware law expansion: add full laws_de candidates that were not in the BGE core.
    # This is the main recall fix for hidden-test citations absent from train/val gold.
    if use_law_expansion and "query_aware_law_expansion_scores" in globals():
        for cit, s in query_aware_law_expansion_scores(query):
            cit = str(cit).strip()
            if cit in BLACKLIST:
                continue
            score_map[cit] = score_map.get(cit, 0.0) + law_expansion_weight * float(s)

        # Conservative court expansion: exact case mentions only.
        for cit, s in query_aware_court_direct_scores(query):
            cit = str(cit).strip()
            if cit in BLACKLIST:
                continue
            score_map[cit] = score_map.get(cit, 0.0) + law_expansion_weight * float(s)


    # v4-safe citation-only rescue:
    # Add only explicit citation candidates. Existing candidates receive at most a tiny tie-break boost.
    if use_safe_citation_rescue and "conservative_citation_rescue_scores" in globals():
        for cit, s in conservative_citation_rescue_scores(query):
            cit = str(cit).strip()
            if cit in BLACKLIST:
                continue
            rescue_score = safe_rescue_weight * float(s)
            if cit in score_map:
                # Existing candidates should not be strongly re-ranked by rescue.
                score_map[cit] = score_map.get(cit, 0.0) + 0.15 * rescue_score
            else:
                # New candidates enter as low-confidence rescue candidates only.
                score_map[cit] = rescue_score

    # v4-safe verifier:
    # Tiny candidate-level tie-breaker for law/article/Abs. matches already visible in the query.
    if use_safe_verifier and "safe_citation_verifier_score" in globals():
        for cit in list(score_map.keys()):
            v = safe_citation_verifier_score(query, cit)
            if v > 0:
                score_map[cit] = score_map.get(cit, 0.0) + safe_verifier_weight * float(v)

    ranked = sorted(score_map.keys(), key=lambda c: -score_map[c])

    # v5-safe tail replacement:
    # keep the original head stable and only replace the last few citations
    # with high-confidence citation-only rescue candidates.
    if use_safe_tail_replace and "safe_tail_replace_results" in globals():
        return safe_tail_replace_results(
            query,
            ranked,
            top_k=top_k,
            keep_topn=safe_tail_keep_topn,
            add_topn=safe_tail_add_topn,
            min_rescue_score=safe_tail_min_score,
        )

    results = []
    seen = set()
    for cit in ranked:
        if cit not in seen:
            results.append(cit)
            seen.add(cit)
        if len(results) >= top_k:
            break

    return results



In [18]:
# Optional citation boost debug cell
# Run this after feature/citation cells to confirm parsing and soft boost trigger.
for i in range(min(5, len(val))):
    q = val.loc[i, "query"]
    parsed = extract_citation(q)
    print("QUERY:", q[:160])
    print("PARSED:", parsed)

    sample_preds = retrieve_final(
        q,
        top_k=10,
        bge_topk=20,
        dense_topn=200,
        dense_weight=0.80,
        lexical_weight=0.20,
        use_features=True,
        feature_topn=180,
        feature_weight=0.70,
        use_citation_boost=True,
        citation_boost_weight=1.00,
    )
    print("PRED:", sample_preds)
    print("=" * 80)


QUERY: May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the principle of
PARSED: {'numbers': {'22', '1'}, 'laws': {'StPO'}}



Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.94it/s]


PRED: ['Art. 221 Abs. 1 StPO', 'Art. 221 Abs. 2 StPO', '1B_210/2023 E. 4.1', '1B_536/2018 E. 5.1', '1B_572/2021 E. 2.1', '1B_581/2022 E. 2.1.2', '1B_211/2017 E. 2.1', '1B_88/2022 E. 2.1', '1B_195/2022 E. 2.2.1', 'BGE 133 I 168 E. 4.1']
QUERY: A claimant holding a national vocational diploma in warehouse operations worked intermittently as a storage technician from 10 March to 20 September 2022 and wa
PARSED: {'numbers': {'1', '17'}, 'laws': {'IVG'}}



Chunks: 100%|██████████| 1/1 [00:00<00:00, 16.61it/s]


PRED: ['Art. 17 Abs. 1 IVG', 'Art. 8 Abs. 1 IVG', 'Art. 28 Abs. 1 IVG', '9C_623/2020 E. 4.2', 'Art. 1 Abs. 1 IVG', 'BGE 139 V 399 E. 5.5', 'BGE 124 V 108 E. 2b', 'Art. 21 Abs. 4 ATSG', 'Art. 69 Abs. 1 IVG', 'Art. 29 Abs. 1 IVG']
QUERY: A. Rivera, a Peruvian national born in 1994 and with no prior convictions in the forum state, is accused of having, between 5 March and 9 March 2024, together w
PARSED: {'numbers': {'22', '1'}, 'laws': {'StPO'}}


...

PRED: ['1B_210/2023 E. 4.1', '1B_572/2021 E. 2.1', '1B_536/2018 E. 5.1', '1B_88/2022 E. 2.1', 'Art. 221 Abs. 1 StPO', 'Art. 428 Abs. 1 StPO', '1B_211/2017 E. 2.1', 'Art. 82 Abs. 1 StPO', 'Art. 396 Abs. 1 StPO', 'Art. 385 Abs. 1 StPO']
QUERY: Mr. Dalton, born in 1941 and resident in a small lakeside town near Thun, executed a handwritten will on 10 October 1997 stating that he left his entire estate 
PARSED: {'numbers': set(), 'laws': set()}



Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.11it/s]


PRED: ['BGE 131 III 601 E. 3.1', 'BGE 131 III 106 E. 1.1', 'Art. 462 ZGB', 'Art. 470 Abs. 1 ZGB', 'Art. 141 Abs. 2 ParlG', 'Art. 963 Abs. 1 ZGB', 'BGE 128 III 419 E. 2.2', 'Art. 965 Abs. 3 ZGB', 'Art. 655 Abs. 2 ZGB', 'Art. 8 ZGB']
QUERY: A parent, separated from their co-parent since 2008, has not had custody of the two children (primary custody is with the other parent); until March 2020 the pa
PARSED: {'numbers': {'1', '20'}, 'laws': set()}



Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.74it/s]


PRED: ['BGE 131 III 209 E. 5', '1B_192/2022 E. 4.1.2', 'BGE 130 III 585 E. 2.2.1', 'BGE 130 III 585 E. 2.1', 'Art. 963 Abs. 1 ZGB', 'Art. 15 Abs. 2 JStG', 'Art. 1 Abs. 2 ZGB', 'Art. 100 Abs. 1 BGG', 'Art. 1 Abs. 2 IPRG', 'Art. 9 Abs. 3 JStG']


In [19]:
# Old external-keyword-retriever experimental cells were intentionally removed.
# This notebook now uses only BGE-M3 retrieval plus the baseline6-style feature engineering layer above.


In [20]:
for i in range(min(5, len(test))):
    q = test.loc[i, "query"]
    preds = retrieve_final(
        q,
        top_k=10,
        bge_topk=10,
        dense_topn=150,
        dense_weight=0.80,
        lexical_weight=0.20,
        use_features=True,
        feature_topn=100,
        feature_weight=0.70,
    )

    print("=" * 100)
    print("query_id:", test.loc[i, "query_id"])
    print("query:", q[:700])
    print("pred:", preds)



Chunks: 100%|██████████| 1/1 [00:00<00:00, 20.98it/s]


query_id: test_001
query: Four U.S.-based software companies (NorthWave Inc., Orion Systems LLC, ClearPeak Corp. and HarborSoft Ltd.) assert that R. Silva, who until April 2019 worked in the Milan development unit of Orion Systems and thereafter was hired by the Swiss firm Lumen (CH) Sàrl, copied the source code and confidential documentation for three programs (codenamed A1, B3 and B4) and that Lumen (CH) Sàrl intends to exploit those materials. The companies ask the cantonal authorities in Lausanne for interim relief consisting of an injunction prohibiting use, reproduction or disclosure of the programs and the immediate seizure and forensic inspection of computers and storage media located at Lumen (CH) Sàrl and i
pred: ['BGE 128 III 76 E. 1b', 'Art. 55 Abs. 1 USG', 'Art. 62 Abs. 1 URG', 'Art. 963 Abs. 1 ZGB', 'Art. 965 Abs. 3 ZGB', 'Art. 655 Abs. 2 ZGB', 'Art. 8 ZGB', 'Art. 12 Abs. 1 NHG', 'Art. 527 ZGB', 'Art. 965 Abs. 2 ZGB']



Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.34it/s]


query_id: test_002
query: On 9 August 2011 a 62‑year‑old cyclist (the claimant) was struck at a junction inside an industrial estate by a delivery van that had been leaving a side access to turn left onto a county road; the cyclist sustained a head injury, a cervical sprain and later had his loss of earning capacity assessed at 45% by the State Disability Office (decision of 10 November 2014). The cyclist brought an action against the van driver and the insurer on 15 October 2013 for damages and enlarged his claims on 12 March 2015 and 20 April 2016 to include past lost earnings and impairment of his economic future. The defendants maintain that these later claims are time‑barred under Art. 83 SVG because the claiman
pred: ['Art. 59 Abs. 1 SVG', 'Art. 59 Abs. 2 SVG', 'Art. 59 Abs. 3 SVG', 'Art. 61 Abs. 1 SVG', 'BGE 132 III 321 E. 2.2.1', 'Art. 58 Abs. 2 SVG', 'BGE 137 III 539 E. 5.2', 'BGE 121 IV 207 E. 2a', 'Art. 83 Abs. 1 SVG', 'Art. 83 Abs. 2 SVG']



Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.55it/s]


query_id: test_003
query: On 12 March 2012, Meridian Leasing Ltd and Orion Transport LLC entered into a three-year vehicle hire agreement for a box truck explicitly designated for business operations; Mr. Carter, the managing partner of Orion Transport, signed the contract personally as a joint and several obligor. After Orion Transport entered insolvency on 30 September 2012, Mr. Carter delivered the truck back to Meridian on 15 October 2012; Meridian inspected the vehicle on 22 November 2012 and invoiced refurbishment expenses of CHF 9,200 (CHF 9,000 of which were described as wear beyond the normal degree). Meridian terminated the lease on 10 December 2012 and sought CHF 18,500; by letter dated 5 January 2013 Meri
pred: ['BGE 127 III 147 E. 2c', 'Art. 336a Abs. 3 OR', 'Art. 20 Abs. 2 OR', 'Art. 659 Abs. 2 OR', 'Art. 24 Abs. 1 OR', 'Art. 41 Abs. 1 OR', 'Art. 398 Abs. 2 OR', 'Art. 20 Abs. 1 OR', 'Art. 659 Abs. 1 OR', 'Art. 216c Abs. 1 OR']



Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.31it/s]


query_id: test_004
query: A publicly listed manufacturing company, Orion Manufacturing AG, was served with a payment order on 14 January 2022 relating to several outstanding invoices, against which it entered a total opposition. The creditor, Delta Supplies Ltd., applied to have the opposition lifted on 30 January 2022 and obtained an order lifting the opposition; the operative part of that order was rendered on 5 September 2022 and the reasons were notified on 20 November 2022. Delta Supplies subsequently filed a petition to place Orion Manufacturing into bankruptcy on 1 February 2024. Orion Manufacturing contests the validity of that bankruptcy petition, contending that the peremptory fifteen‑month period under Art
pred: ['Art. 166 Abs. 2 SchKG', 'Art. 82 Abs. 1 SchKG', 'BGE 127 III 147 E. 2c', 'Art. 83 Abs. 2 SchKG', 'Art. 69 Abs. 2 SchKG', 'Art. 166 Abs. 1 SchKG', 'BGE 148 V 21 E. 5.3', 'Art. 9 Abs. 2 VZG', 'Art. 325 Abs. 2 ZPO', 'Art. 210 Abs. 1 OR']



Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.68it/s]


query_id: test_005
query: A logistics company (R Ltd.), which owns a distribution warehouse, its facility manager (S Ltd.), and the warehouse’s sole shareholder-director (T) signed on 14 March 2012 a written “mortgage lending arrangement” under which S Ltd. extended credit to R Ltd. (and to T and U Ltd.) and took as security a bearer mortgage certificate for CHF 420,000 encumbering plot no. 2207; the certificate, created on 20 March 2012, contains no designation of any obligor and was handed over in full ownership to S Ltd. Various management advances were posted to an account producing an alleged debit balance of CHF 401,237.88; S Ltd. issued a formal demand and on 9 September 2013 denounced the bearer certificate w
pred: ['BGE 146 III 326 E. 6.1', 'BGE 127 III 543 E. 2c', 'Art. 963 Abs. 1 ZGB', '4A_379/2016 E. 3.3.1', 'Art. 1 Abs. 2 ZGB', 'Art. 1 Abs. 2 IPRG', 'Art. 20 Abs. 2 OR', 'Art. 946 Abs. 1 ZGB', 'Art. 969 Abs. 1 ZGB', 'BGE 132 III 449 E. 2']


In [21]:
for i in range(min(5, len(val))):
    q = val.loc[i, "query"]
    gold = split_citations(val.loc[i, "gold_citations"])
    pred = retrieve_final(
        q,
        top_k=10,
        bge_topk=10,
        dense_topn=150,
        dense_weight=0.80,
        lexical_weight=0.20,
        use_features=True,
        feature_topn=100,
        feature_weight=0.70,
    )

    print("=" * 100)
    print("query_id:", val.loc[i, "query_id"])
    print("QUERY:", q[:700])
    print("GOLD :", gold)
    print("PRED :", pred)



Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.00it/s]


query_id: val_001
QUERY: May a court lawfully order a three‑month extension of pre‑trial detention under Art. 221 Abs. 1 lit. b StPO (risk of collusion) consistent with the principle of proportionality when the accused—detained after an alleged late‑night assault and theft of a courier satchel containing, inter alia, €5,600—was remanded by an order dated 18 October 2024 for a maximum period up to 15 January 2025, the prosecutor sought an extension on 10 December 2024 primarily citing a concrete risk that the detainee would influence witnesses or tamper with evidence and a risk of reoffending, while the detainee opposed the extension on the ground that most witnesses have already been interviewed, the investigative s
GOLD : ['Art. 221 Abs. 1 StPO', 'Art. 140 Abs. 1 StGB', 'Art. 396 Abs. 1 StPO', 'Art. 222 StPO', 'Art. 393 Abs. 1 StPO', 'Art. 382 Abs. 1 StPO', 'Art. 385 Abs. 1 StPO', 'Art. 221 Abs. 2 StPO', 'Art. 227 Abs. 1 StPO', 'Art. 212 Abs. 3 StPO', 'Art. 390 Abs. 2 StPO', 'Art. 422


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.25it/s]


query_id: val_002
QUERY: A claimant holding a national vocational diploma in warehouse operations worked intermittently as a storage technician from 10 March to 20 September 2022 and was entered as job-seeking on 1 October 2022. From mid-2021 onwards he has suffered from a chronic allergic respiratory disorder (eosinophilic allergic asthma with nasal/ocular symptoms) with raised IgE to seasonal pollens and household mites; he required inpatient treatment for an acute exacerbation in August 2022. Allergy specialists advised placement in a temperature-controlled, low-dust workplace, yet an occupational pulmonologist’s assessment dated 15 April 2023 concluded the claimant retained full capacity for employment and that 
GOLD : ['Art. 8 Abs. 1 ATSG', 'Art. 8 Abs. 1 IVG', 'Art. 17 Abs. 1 IVG', 'Art. 1 Abs. 1 IVG', 'Art. 56 Abs. 1 ATSG', 'Art. 69 Abs. 1 IVG', 'Art. 60 Abs. 1 ATSG', 'Art. 61 ATSG', 'Art. 29 Abs. 1 IVG', 'Art. 4 Abs. 1 IVG', 'Art. 6 ATSG', 'Art. 8 Abs. 3 IVG', 'Art. 18d IVG', '


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.60it/s]


query_id: val_003
QUERY: A. Rivera, a Peruvian national born in 1994 and with no prior convictions in the forum state, is accused of having, between 5 March and 9 March 2024, together with three accomplices (B. L., C. M. and D. S.) taken part in a series of offenses including the theft and driving off of delivery vans, forcible entry into a riverside storage unit and the theft of items (notably a replica handgun), the assault of a witness with a metal bar, the discharge of a shot toward the river, and on 9 March 2024 the theft of a further van followed by a high‑speed pursuit in which shots were fired toward law‑enforcement officers; A. Rivera was located in a neighbouring state on 18 March 2024, returned to the pro
GOLD : ['Art. 29 Abs. 2 BV', 'Art. 221 Abs. 1 StPO', 'Art. 393 Abs. 1 StPO', 'Art. 222 StPO', 'Art. 384 StPO', 'Art. 396 Abs. 1 StPO', 'Art. 382 Abs. 1 StPO', 'Art. 385 Abs. 1 StPO', 'Art. 3 Abs. 2 StPO', 'Art. 390 Abs. 2 StPO', 'Art. 422 Abs. 1 StPO', 'Art. 422 Abs. 2 StPO


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.68it/s]


query_id: val_004
QUERY: Mr. Dalton, born in 1941 and resident in a small lakeside town near Thun, executed a handwritten will on 10 October 1997 stating that he left his entire estate to his partner Ms. Lang and, should she predecease him, to his granddaughters Anna (born 1988) and Bella (born 1991). Mr. Dalton and Ms. Lang died in 2010 in an alpine avalanche; Dalton's surviving half‑siblings and a cousin contest the validity of the 10 October 1997 handwritten instrument, alleging that Dalton did not have sufficient command of German and therefore could not have composed a testamentary text containing legal terms such as bequeath or legatee, so the document does not reflect his true intention. Under the Civil Code,
GOLD : ['Art. 505 Abs. 1 ZGB', 'Art. 467 ZGB', 'Art. 469 Abs. 1 ZGB', 'Art. 471 ZGB', 'Art. 20 Abs. 2 OR', 'Art. 520a ZGB', 'Art. 469 Abs. 2 ZGB', 'Art. 458 Abs. 3 ZGB', 'Art. 100 Abs. 1 BGG', 'BGE 131 III 601 E. 3.1']
PRED : ['BGE 131 III 601 E. 3.1', 'BGE 131 III 106 E. 1


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.40it/s]


query_id: val_005
QUERY: A parent, separated from their co-parent since 2008, has not had custody of the two children (primary custody is with the other parent); until March 2020 the parent exercised a longstanding visitation pattern (Wednesday evenings, alternate weekends including overnight stays, and portions of school holidays). In March 2021 the parent was accused of twice engaging in sexually inappropriate touching of a teenage friend of the older child (then 15), was briefly held in custody for four days and is the subject of ongoing criminal proceedings and a forensic psychiatric assessment whose conclusions have not been placed in the family-file; the parent says alcohol was involved and since May 2021 has 
GOLD : ['Art. 133 Abs. 1 ZGB', 'Art. 133 Abs. 2 ZGB', 'Art. 285 Abs. 1 ZGB', 'Art. 100 Abs. 1 BGG', 'BGE 131 III 209 E. 5', 'BGE 128 III 411 E. 3.2.1', 'Art. 274 Abs. 2 ZGB', 'BGE 130 III 585 E. 2.1', 'BGE 130 III 585 E. 2.2.1', 'Art. 273 Abs. 1 ZGB', 'BGE 126 III 219 E. 2'

In [22]:
def evaluate(
    top_k=10,
    bge_topk=10,
    dense_topn=150,
    dense_weight=0.80,
    lexical_weight=0.20,
    use_features=True,
    feature_topn=120,
    feature_weight=0.70,
    use_citation_boost=True,
    citation_boost_weight=1.00,
    use_law_expansion=True,
    law_expansion_weight=0.65,
    use_safe_citation_rescue=False,
    safe_rescue_weight=0.10,
    use_safe_verifier=False,
    safe_verifier_weight=0.05,
    use_safe_tail_replace=False,
    safe_tail_keep_topn=22,
    safe_tail_add_topn=3,
    safe_tail_min_score=0.55,
):
    scores = []

    for i in tqdm(range(len(val)), desc="Validating on val"):
        q = val.loc[i, "query"]
        gold = split_citations(val.loc[i, "gold_citations"])

        pred = retrieve_final(
            q,
            top_k=top_k,
            bge_topk=bge_topk,
            dense_topn=dense_topn,
            dense_weight=dense_weight,
            lexical_weight=lexical_weight,
            use_features=use_features,
            feature_topn=feature_topn,
            feature_weight=feature_weight,
            use_citation_boost=use_citation_boost,
            citation_boost_weight=citation_boost_weight,
            use_law_expansion=use_law_expansion,
            law_expansion_weight=law_expansion_weight,
            use_safe_citation_rescue=use_safe_citation_rescue,
            safe_rescue_weight=safe_rescue_weight,
            use_safe_verifier=use_safe_verifier,
            safe_verifier_weight=safe_verifier_weight,
            use_safe_tail_replace=use_safe_tail_replace,
            safe_tail_keep_topn=safe_tail_keep_topn,
            safe_tail_add_topn=safe_tail_add_topn,
            safe_tail_min_score=safe_tail_min_score,
        )

        scores.append(f1_score_set(pred, gold))

    return np.mean(scores), np.mean(np.array(scores) > 0)



In [23]:
mean_f1, non_zero = evaluate(
    top_k=10,
    bge_topk=10,
    dense_topn=150,
    dense_weight=0.80,
    lexical_weight=0.20,
    use_features=True,
    feature_topn=120,
    feature_weight=0.70,
)

print("Mean val F1:", mean_f1)
print("Non-zero ratio:", non_zero)


Validating on val:   0%|          | 0/10 [00:00<?, ?it/s]



Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.08it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.23it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.50it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.46it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.35it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.25it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 14.27it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.36it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.38it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 16.14it/s]


Mean val F1: 0.20158461399722344
Non-zero ratio: 1.0


In [24]:
settings = [
    # top_k, bge_topk, dense_topn, dense_w, lexical_w, use_features, feature_topn, feature_w
    (10, 10, 100, 0.85, 0.15, True, 80, 0.50),
    (10, 10, 150, 0.80, 0.20, True, 120, 0.70),
    (15, 15, 150, 0.80, 0.20, True, 150, 0.70),
    (25, 20, 200, 0.80, 0.20, True, 180, 0.70),
    # Pure BGE fallback, for comparison:
    (10, 10, 150, 0.80, 0.20, False, 0, 0.00),
]

for top_k, bge_topk, dense_topn, dw, lw, use_fe, fe_topn, fe_w in settings:
    mean_f1, nz = evaluate(
        top_k=top_k,
        bge_topk=bge_topk,
        dense_topn=dense_topn,
        dense_weight=dw,
        lexical_weight=lw,
        use_features=use_fe,
        feature_topn=fe_topn,
        feature_weight=fe_w,
    )

    print(
        f"top_k={top_k} | bge_topk={bge_topk} | dense_topn={dense_topn} | "
        f"dense={dw} | lexical={lw} | features={use_fe} | "
        f"feature_topn={fe_topn} | feature_w={fe_w} | F1={mean_f1:.5f} | nz={nz:.2f}"
    )


Validating on val:   0%|          | 0/10 [00:00<?, ?it/s]



Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.51it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.48it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.69it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.28it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.78it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.38it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 14.26it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.42it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 23.15it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.70it/s]


top_k=10 | bge_topk=10 | dense_topn=100 | dense=0.85 | lexical=0.15 | features=True | feature_topn=80 | feature_w=0.5 | F1=0.22502 | nz=1.00


Validating on val:   0%|          | 0/10 [00:00<?, ?it/s]



Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.15it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.51it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.72it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.49it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.51it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.36it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 14.22it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.04it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 23.36it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.63it/s]


...



Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.17it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.03it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.65it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.75it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.64it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 16.99it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 14.11it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.36it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 20.96it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.67it/s]


top_k=25 | bge_topk=20 | dense_topn=200 | dense=0.8 | lexical=0.2 | features=True | feature_topn=180 | feature_w=0.7 | F1=0.24696 | nz=1.00


Validating on val:   0%|          | 0/10 [00:00<?, ?it/s]



Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.58it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.69it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.73it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.04it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.77it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 18.26it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 14.18it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 18.03it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 24.46it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.76it/s]


top_k=10 | bge_topk=10 | dense_topn=150 | dense=0.8 | lexical=0.2 | features=False | feature_topn=0 | feature_w=0.0 | F1=0.23426 | nz=1.00


In [25]:

# ============================================================
# Final settings
# ============================================================
# Reranker uses a wider candidate pool internally, but final output should stay at 10
# unless you have confirmed leaderboard prefers more citations.
BEST_TOP_K = 25
BEST_BGE_TOPK = 20
BEST_DENSE_TOPN = 200
BEST_DENSE_WEIGHT = 0.8
BEST_LEXICAL_WEIGHT = 0.2
BEST_USE_FEATURES = True
BEST_FEATURE_TOPN = 180
BEST_FEATURE_WEIGHT = 0.7

BEST_USE_CITATION_BOOST = True
BEST_CITATION_BOOST_WEIGHT = 1.00

# New recall-oriented expansion. Keep it moderate: it adds unseen laws_de citations
# but does not replace the original baseline scoring.
BEST_USE_LAW_EXPANSION = True
BEST_LAW_EXPANSION_WEIGHT = 0.75

# Reranker was consistently worse on LB/val for this branch, so keep submission baseline-only.
USE_SUPERVISED_RERANKER_FOR_SUBMISSION = False

# Optional LightGBM learning-to-rank style reranker branch.
# It uses the existing retrieve_final() as candidate generator, then trains a small table model on train/val labels.
# Set False to fall back to the original 0.08364 submission path.
USE_LGBM_RERANKER_FOR_SUBMISSION = True
LGBM_CANDIDATE_TOPN = 150
LGBM_TRANSFER_TOPK = 12
LGBM_TRANSFER_TOPCIT = 80



# v4-safe conservative additions.
# Keep these small: they should not dominate the original 0.07802 backbone.
BEST_USE_SAFE_CITATION_RESCUE = True
BEST_SAFE_RESCUE_WEIGHT = 0.10
BEST_USE_SAFE_VERIFIER = True
BEST_SAFE_VERIFIER_WEIGHT = 0.05

# v5-safe tail replacement.
# This is the main v5 change: it can actually change the final citation set,
# but only by replacing the tail and only with explicit citation-pattern rescue.
BEST_USE_SAFE_TAIL_REPLACE = True
BEST_SAFE_TAIL_KEEP_TOPN = 20
BEST_SAFE_TAIL_ADD_TOPN = 5
BEST_SAFE_TAIL_MIN_SCORE = 0.65

# v6 uses an overridden safe_tail_replace_results() defined above.
# Try 20/5/0.65 as an aggressive follow-up if 21/4/0.60 improves LB.

# Optional ablation switch. Keep False for normal submission generation.
RUN_SAFE_V4_ABLATION = False


In [26]:
# ============================================================
# Optional v4-safe ablation
# ------------------------------------------------------------
# Set RUN_SAFE_V4_ABLATION = True in the settings cell if you want to compare:
#   1) original 0.07802-style settings
#   2) v4-safe citation rescue + tiny verifier
# ============================================================

if RUN_SAFE_V4_ABLATION:
    old_f1, old_nz = evaluate(
        top_k=BEST_TOP_K,
        bge_topk=BEST_BGE_TOPK,
        dense_topn=BEST_DENSE_TOPN,
        dense_weight=BEST_DENSE_WEIGHT,
        lexical_weight=BEST_LEXICAL_WEIGHT,
        use_features=BEST_USE_FEATURES,
        feature_topn=BEST_FEATURE_TOPN,
        feature_weight=BEST_FEATURE_WEIGHT,
        use_citation_boost=BEST_USE_CITATION_BOOST,
        citation_boost_weight=BEST_CITATION_BOOST_WEIGHT,
        use_law_expansion=BEST_USE_LAW_EXPANSION,
        law_expansion_weight=BEST_LAW_EXPANSION_WEIGHT,
        use_safe_citation_rescue=False,
        use_safe_verifier=False,
        use_safe_tail_replace=False,
    )

    new_f1, new_nz = evaluate(
        top_k=BEST_TOP_K,
        bge_topk=BEST_BGE_TOPK,
        dense_topn=BEST_DENSE_TOPN,
        dense_weight=BEST_DENSE_WEIGHT,
        lexical_weight=BEST_LEXICAL_WEIGHT,
        use_features=BEST_USE_FEATURES,
        feature_topn=BEST_FEATURE_TOPN,
        feature_weight=BEST_FEATURE_WEIGHT,
        use_citation_boost=BEST_USE_CITATION_BOOST,
        citation_boost_weight=BEST_CITATION_BOOST_WEIGHT,
        use_law_expansion=BEST_USE_LAW_EXPANSION,
        law_expansion_weight=BEST_LAW_EXPANSION_WEIGHT,
        use_safe_citation_rescue=BEST_USE_SAFE_CITATION_RESCUE,
        safe_rescue_weight=BEST_SAFE_RESCUE_WEIGHT,
        use_safe_verifier=BEST_USE_SAFE_VERIFIER,
        safe_verifier_weight=BEST_SAFE_VERIFIER_WEIGHT,
    )

    print(f"OLD  | F1={old_f1:.5f} | nonzero={old_nz:.3f}")
    print(f"V4   | F1={new_f1:.5f} | nonzero={new_nz:.3f}")


In [27]:
# ============================================================
# Old supervised pointwise reranker is intentionally skipped
# ------------------------------------------------------------
# Previous experiments showed the supervised reranker hurt validation/LB.
# This branch focuses only on recall expansion:
#   baseline BGE core + full-laws query-aware expansion + conservative court direct hits.
# ============================================================

if USE_SUPERVISED_RERANKER_FOR_SUBMISSION:
    raise RuntimeError(
        "USE_SUPERVISED_RERANKER_FOR_SUBMISSION is True, but this expansion branch disables the old reranker. "
        "Set it to False or restore the old reranker cell."
    )

print("Supervised reranker skipped. Using baseline + query-aware law expansion + v5 safe tail replacement for submission.")


Supervised reranker skipped. Using baseline + query-aware law expansion + v5 safe tail replacement for submission.


In [28]:
V6_TAIL_COCIT_SEED_TOPN = 5
V6_TAIL_COCIT_TOP_EACH = 3
V6_TAIL_COCIT_WEIGHT = 0.18

V6_ADJ_ARTICLE_WINDOW = 1
V6_ADJ_ARTICLE_WEIGHT = 0.22
V6_ABS_MATCH_WEIGHT = 0.35
V6_ABS_MISMATCH_PENALTY = -0.50
V6_FREQ_BIAS_WEIGHT = 0.08
V6_RESTRICT_TAIL_TO_QUERY_OR_HEAD_LAW = True

In [29]:
LGBM_TRANSFER_TOPK = 12
LGBM_TRANSFER_TOPCIT = 25
LGBM_CANDIDATE_TOPN = 180

In [30]:
# ============================================================
# LightGBM reranker branch with stronger transfer features
# ============================================================

if USE_LGBM_RERANKER_FOR_SUBMISSION:
    # --- FIX: define missing helper ---
    def _article_to_int(art):
        import re
        m = re.match(r'^(\d+)', str(art))
        return int(m.group(1)) if m else None

    # --- fallback for missing globals ---
    if 'law_expansion_exact' not in globals():
        law_expansion_exact = {}
    if 'law_expansion_parts' not in globals():
        law_expansion_parts = {}

    import sys, subprocess, importlib.util
    if importlib.util.find_spec("lightgbm") is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    import lightgbm as lgb
    from collections import defaultdict, Counter
    from sklearn.metrics.pairwise import cosine_similarity

    LGBM_TRANSFER_TOPK = 12
    LGBM_TRANSFER_TOPCIT = 25
    LGBM_CANDIDATE_TOPN = 180

    def _lgb_split_citations(s):
        if pd.isna(s) or str(s).strip() == "":
            return []
        return [x.strip() for x in str(s).split(";") if x.strip()]

    if "gold_list" not in train.columns:
        train["gold_list"] = train["gold_citations"].apply(_lgb_split_citations)
    if "gold_list" not in val.columns:
        val["gold_list"] = val["gold_citations"].apply(_lgb_split_citations)

    _lgb_train_val = pd.concat([
        train[["query_id", "query", "gold_list"]].assign(_split="train"),
        val[["query_id", "query", "gold_list"]].assign(_split="val"),
    ], ignore_index=True)

    _LGB_LAW_RE = re.compile(
        r"\b(ZGB|OR|StGB|StPO|ZPO|BGG|SVG|SchKG|IPRG|AIG|ATSG|KVG|UVG|AHVG|IVG|URG|MSchG|UWG|MWSTG|DBG|VStG|ArG|BVG|PrHG|BV|EMRK)\b",
        flags=re.IGNORECASE,
    )
    _LGB_ART_RE = re.compile(
        r"Art\.?\s*(\d+[a-zA-Z]?)"
        r"(?:\s*Abs\.?\s*(\d+[a-zA-Z]?))?"
        r".*?\b(ZGB|OR|StGB|StPO|ZPO|BGG|SVG|SchKG|IPRG|AIG|ATSG|KVG|UVG|AHVG|IVG|URG|MSchG|UWG|MWSTG|DBG|VStG|ArG|BVG|PrHG|BV|EMRK)\b",
        flags=re.IGNORECASE,
    )
    _LGB_CASE_RE = re.compile(r"\b\d+[A-Z]_[0-9]+/[0-9]{4}|BGE\s+\d+\s+[IVX]+\s+\d+", flags=re.IGNORECASE)

    def _lgb_parse_art(cit):
        m = _LGB_ART_RE.search(str(cit))
        if not m:
            return None
        law = m.group(3).upper()
        art = re.sub(r"\D", "", m.group(1))
        abs_ = m.group(2).lower() if m.group(2) else ""
        return law, art, abs_

    def _lgb_query_laws(q):
        return set(m.group(1).upper() for m in _LGB_LAW_RE.finditer(str(q)))

    def _lgb_query_arts(q):
        out = []
        for m in _LGB_ART_RE.finditer(str(q)):
            law = m.group(3).upper()
            art = re.sub(r"\D", "", m.group(1))
            abs_ = m.group(2).lower() if m.group(2) else ""
            out.append((law, art, abs_))
        return out

    def _lgb_has_case_signal(q):
        return int(bool(_LGB_CASE_RE.search(str(q))))

    def _lgb_get_candidates(query, topn=LGBM_CANDIDATE_TOPN):
        cands = retrieve_final(
            query,
            top_k=topn,
            bge_topk=max(BEST_BGE_TOPK, min(80, topn)),
            dense_topn=max(BEST_DENSE_TOPN, topn),
            dense_weight=BEST_DENSE_WEIGHT,
            lexical_weight=BEST_LEXICAL_WEIGHT,
            use_features=BEST_USE_FEATURES,
            feature_topn=max(BEST_FEATURE_TOPN, topn),
            feature_weight=BEST_FEATURE_WEIGHT,
            use_citation_boost=BEST_USE_CITATION_BOOST,
            citation_boost_weight=BEST_CITATION_BOOST_WEIGHT,
            use_law_expansion=BEST_USE_LAW_EXPANSION,
            law_expansion_weight=BEST_LAW_EXPANSION_WEIGHT,
            use_safe_citation_rescue=BEST_USE_SAFE_CITATION_RESCUE,
            safe_rescue_weight=BEST_SAFE_RESCUE_WEIGHT,
            use_safe_verifier=BEST_USE_SAFE_VERIFIER,
            safe_verifier_weight=BEST_SAFE_VERIFIER_WEIGHT,
            use_safe_tail_replace=False,
        )
        out, seen = [], set()
        for c in cands:
            c = str(c).strip()
            if c and c not in seen and c not in BLACKLIST:
                out.append(c)
                seen.add(c)
            if len(out) >= topn:
                break
        return out

    def _lgb_build_query_embeddings():
        ref_queries = _lgb_train_val["query"].astype(str).map(normalize_text).tolist()
        all_queries = pd.concat([
            train[["query_id", "query"]].assign(_split="train"),
            val[["query_id", "query"]].assign(_split="val"),
            test[["query_id", "query"]].assign(_split="test"),
        ], ignore_index=True)

        q_texts = all_queries["query"].astype(str).map(normalize_text).tolist()

        ref_emb = model.encode(
            ref_queries,
            batch_size=16,
            max_length=512,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )["dense_vecs"].astype("float32")

        all_emb = model.encode(
            q_texts,
            batch_size=16,
            max_length=512,
            return_dense=True,
            return_sparse=False,
            return_colbert_vecs=False,
        )["dense_vecs"].astype("float32")

        faiss.normalize_L2(ref_emb)
        faiss.normalize_L2(all_emb)

        return all_queries, all_emb, ref_emb

    def _lgb_make_transfer_maps():
        all_queries, all_emb, ref_emb = _lgb_build_query_embeddings()
        sims = all_emb @ ref_emb.T

        ref_gold = _lgb_train_val["gold_list"].tolist()
        ref_qids = _lgb_train_val["query_id"].astype(str).tolist()

        maps = {}

        for i, row in all_queries.iterrows():
            qid = str(row["query_id"])

            scores = defaultdict(float)
            hit_count = Counter()
            best_sim = defaultdict(float)

            idxs = np.argsort(-sims[i])[: max(LGBM_TRANSFER_TOPK + 8, LGBM_TRANSFER_TOPK)]

            used = 0
            for j in idxs:
                if ref_qids[j] == qid:
                    continue

                sim = float(sims[i, j])
                if sim <= 0:
                    continue

                for cit in ref_gold[j][:LGBM_TRANSFER_TOPCIT]:
                    cit = str(cit).strip()
                    if not cit:
                        continue

                    scores[cit] += sim
                    hit_count[cit] += 1
                    best_sim[cit] = max(best_sim[cit], sim)

                used += 1
                if used >= LGBM_TRANSFER_TOPK:
                    break

            if scores:
                mx = max(scores.values())
                if mx > 0:
                    for k in list(scores.keys()):
                        scores[k] /= mx

            maps[qid] = {
                k: {
                    "score": float(scores.get(k, 0.0)),
                    "hit_count": int(hit_count.get(k, 0)),
                    "best_sim": float(best_sim.get(k, 0.0)),
                }
                for k in set(scores) | set(hit_count) | set(best_sim)
            }

        return maps

    print("Building LightGBM transfer maps...")
    _lgb_transfer_by_qid = _lgb_make_transfer_maps()

    _lgb_co = defaultdict(Counter)
    for golds in _lgb_train_val["gold_list"].tolist():
        for a in golds:
            for b in golds:
                if a != b:
                    _lgb_co[str(a).strip()][str(b).strip()] += 1

    def _lgb_graph_scores(seed_cands, top_seed=10):
        scores = defaultdict(float)

        for r, seed in enumerate(seed_cands[:top_seed]):
            seed_weight = 1.0 / (r + 1)

            for nb, cnt in sorted(_lgb_co.get(seed, {}).items(), key=lambda x: x[1], reverse=True)[:12]:
                scores[nb] += seed_weight * min(float(cnt) / 5.0, 1.0)

        if scores:
            mx = max(scores.values())
            if mx > 0:
                for k in list(scores.keys()):
                    scores[k] /= mx

        return scores

    def _lgb_build_rows(df, split_name, topn=LGBM_CANDIDATE_TOPN, include_gold=True):
        rows = []
        has_gold = "gold_list" in df.columns

        for _, row in tqdm(df.iterrows(), total=len(df), desc=f"LGBM features {split_name}"):
            qid = str(row["query_id"])
            query = str(row["query"])

            cands = _lgb_get_candidates(query, topn=topn)

            golds = [str(x).strip() for x in row["gold_list"]] if has_gold else []
            gold_set = set(golds)

            if include_gold and has_gold:
                for g in golds:
                    if g and g not in cands and g not in BLACKLIST:
                        cands.append(g)

            transfer = _lgb_transfer_by_qid.get(qid, {})

            for cit, info in sorted(
                transfer.items(),
                key=lambda x: x[1]["score"] if isinstance(x[1], dict) else float(x[1]),
                reverse=True,
            )[:70]:
                if cit not in cands and cit not in BLACKLIST:
                    cands.append(cit)

            graph = _lgb_graph_scores(cands)

            q_laws = _lgb_query_laws(query)
            q_arts = _lgb_query_arts(query)
            q_has_case = _lgb_has_case_signal(query)

            for rank, cit in enumerate(cands):
                p = _lgb_parse_art(cit)

                same_law = same_article = same_abs = wrong_abs = adjacent_article = 0

                if p:
                    c_law, c_art, c_abs = p
                    same_law = int(c_law in q_laws)

                    for q_law, q_art, q_abs in q_arts:
                        if c_law == q_law and c_art == q_art:
                            same_article = 1

                            if q_abs and c_abs and q_abs == c_abs:
                                same_abs = 1
                            elif q_abs and c_abs and q_abs != c_abs:
                                wrong_abs = 1

                        if c_law == q_law:
                            try:
                                adjacent_article = max(
                                    adjacent_article,
                                    int(abs(int(c_art) - int(q_art)) == 1),
                                )
                            except Exception:
                                pass

                freq_score = 0.0
                if "citation_freq" in globals() and "max_freq" in globals() and max_freq:
                    freq_score = float(citation_freq.get(cit, 0)) / float(max_freq)
                elif "freq_counter" in globals():
                    mx = max(freq_counter.values()) if len(freq_counter) else 1
                    freq_score = float(freq_counter.get(cit, 0)) / float(mx)

                _tinfo = transfer.get(cit, {})
                if not isinstance(_tinfo, dict):
                    _tinfo = {"score": float(_tinfo), "hit_count": 0, "best_sim": 0.0}

                rows.append({
                    "split": split_name,
                    "query_id": qid,
                    "citation": cit,
                    "rank": int(rank),
                    "rank_score": 1.0 / (rank + 1.0),
                    "is_top25": int(rank < 25),
                    "same_law": same_law,
                    "same_article": same_article,
                    "same_abs": same_abs,
                    "wrong_abs": wrong_abs,
                    "adjacent_article": adjacent_article,
                    "query_has_case": q_has_case,
                    "transfer_score": float(_tinfo.get("score", 0.0)),
                    "transfer_hit_count": int(_tinfo.get("hit_count", 0)),
                    "transfer_best_sim": float(_tinfo.get("best_sim", 0.0)),
                    "cocit_score": float(graph.get(cit, 0.0)),
                    "freq_score": float(freq_score),
                    "label": int(cit in gold_set) if has_gold else -1,
                })

        return pd.DataFrame(rows)

    print("Building LightGBM feature tables...")
    _lgb_train_feat = _lgb_build_rows(train, "train", topn=LGBM_CANDIDATE_TOPN, include_gold=True)
    _lgb_val_feat = _lgb_build_rows(val, "val", topn=LGBM_CANDIDATE_TOPN, include_gold=True)
    _lgb_test_feat = _lgb_build_rows(test, "test", topn=LGBM_CANDIDATE_TOPN, include_gold=False)

    LGBM_FEATURES = [
        "rank",
        "rank_score",
        "is_top25",
        "same_law",
        "same_article",
        "same_abs",
        "wrong_abs",
        "adjacent_article",
        "query_has_case",
        "transfer_score",
        "transfer_hit_count",
        "transfer_best_sim",
        "cocit_score",
        "freq_score",
    ]

    print("Training LightGBM reranker...")
    _lgb_model = lgb.LGBMClassifier(
        objective="binary",
        n_estimators=650,
        learning_rate=0.03,
        num_leaves=31,
        max_depth=5,
        subsample=0.85,
        colsample_bytree=0.85,
        min_child_samples=20,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1,
    )

    _lgb_model.fit(
        _lgb_train_feat[LGBM_FEATURES],
        _lgb_train_feat["label"],
        eval_set=[(_lgb_val_feat[LGBM_FEATURES], _lgb_val_feat["label"])],
        eval_metric="binary_logloss",
    )

    _lgb_test_feat["lgb_score"] = _lgb_model.predict_proba(_lgb_test_feat[LGBM_FEATURES])[:, 1]

    def build_lgbm_submission_df(top_k=25):
        rows = []

        for qid, g in _lgb_test_feat.groupby("query_id", sort=False):
            g = g.sort_values(["lgb_score", "rank_score"], ascending=[False, False])

            preds, seen = [], set()

            for cit in g["citation"].astype(str).tolist():
                cit = cit.strip()
                if cit and cit not in seen and cit not in BLACKLIST:
                    preds.append(cit)
                    seen.add(cit)

                if len(preds) >= top_k:
                    break

            rows.append({
                "query_id": qid,
                "predicted_citations": ";".join(preds),
            })

        return pd.DataFrame(rows)

    # ============================================================
    # Build raw LGBM output, then run safe-tail parameter sweep
    # ------------------------------------------------------------
    # IMPORTANT:
    # - lgbm_submission_raw is the original LGBM top-25 output.
    # - Every safe-tail variant starts from lgbm_submission_raw.
    # - Do NOT stack safe-tail postprocess on an already postprocessed output.
    # ============================================================

    lgbm_submission = build_lgbm_submission_df(top_k=BEST_TOP_K)
    lgbm_submission_raw = lgbm_submission.copy()
    print("Raw LGBM submission preview:")
    print(lgbm_submission_raw.head().to_string())

    POST_SAFE_TAIL = True

    def _split_pred_for_safe_tail(s):
        if pd.isna(s) or str(s).strip() == "":
            return []
        return [c.strip() for c in str(s).split(";") if c.strip()]

    def _prediction_count_dist(sub_df):
        return sub_df["predicted_citations"].apply(
            lambda x: len(_split_pred_for_safe_tail(x))
        ).value_counts().sort_index().to_dict()

    def postprocess_lgbm_with_safe_tail_params(
        sub_df,
        keep_topn=23,
        add_topn=2,
        min_rescue_score=0.70,
        top_k=25,
        verbose=True,
    ):
        rows = []
        changed = []
        test_query_map = dict(zip(test["query_id"].astype(str), test["query"].astype(str)))

        for _, row in sub_df.iterrows():
            qid = str(row["query_id"])
            query = test_query_map.get(qid, "")
            old_pred = _split_pred_for_safe_tail(row["predicted_citations"])[:top_k]
            pred = old_pred.copy()

            if POST_SAFE_TAIL and add_topn > 0 and "safe_tail_replace_results" in globals():
                try:
                    pred = safe_tail_replace_results(
                        query,
                        pred,
                        top_k=top_k,
                        keep_topn=keep_topn,
                        add_topn=add_topn,
                        min_rescue_score=min_rescue_score,
                    )
                except Exception as e:
                    print("safe_tail_replace_results failed:", qid, keep_topn, add_topn, min_rescue_score, repr(e))
                    pred = old_pred.copy()

            pred = [str(c).strip() for c in pred if str(c).strip()]
            pred = list(dict.fromkeys(pred))[:top_k]

            if old_pred != pred:
                changed.append((qid, old_pred, pred))

            rows.append({
                "query_id": qid,
                "predicted_citations": ";".join(pred),
            })

        out = pd.DataFrame(rows)
        count_dist = _prediction_count_dist(out)

        if verbose:
            print("changed_rows:", len(changed))
            print("count_dist:", count_dist)
            for qid, old, new in changed[:5]:
                print("-" * 100)
                print("qid:", qid)
                print("old tail:", old[keep_topn:top_k])
                print("new tail:", new[keep_topn:top_k])

        return out, changed, count_dist

    def compare_submissions(a, b, name_a="A", name_b="B", top_k=25):
        rows = []
        a_map = dict(zip(a["query_id"].astype(str), a["predicted_citations"].astype(str)))
        b_map = dict(zip(b["query_id"].astype(str), b["predicted_citations"].astype(str)))

        for qid in sorted(set(a_map) | set(b_map)):
            pa = _split_pred_for_safe_tail(a_map.get(qid, ""))[:top_k]
            pb = _split_pred_for_safe_tail(b_map.get(qid, ""))[:top_k]
            sa, sb = set(pa), set(pb)
            rows.append({
                "query_id": qid,
                f"only_{name_a}": ";".join([x for x in pa if x not in sb]),
                f"only_{name_b}": ";".join([x for x in pb if x not in sa]),
                "same_count": len(sa & sb),
                "changed": int(pa != pb),
            })

        return pd.DataFrame(rows)

    # Parameter grid. The first one reproduces friend's 0.10249 setting.
    SAFE_TAIL_PARAM_GRID = [
        (23, 2, 0.70),
        (24, 1, 0.70),
        (24, 1, 0.75),
        (23, 2, 0.75),
        (23, 2, 0.80),
        (22, 3, 0.70),
        (22, 3, 0.75),
        (25, 0, 999.0),  # raw LGBM / no postprocess baseline
    ]

    sweep_summary = []
    sweep_outputs = {}

    print("\nRunning safe-tail parameter sweep from raw LGBM output...")

    for keep_topn, add_topn, min_score in SAFE_TAIL_PARAM_GRID:
        # Create tag with proper formatting (min_score as string without trailing zeros)
        min_str = f"{min_score:.0f}" if min_score == int(min_score) else str(min_score).rstrip('0').rstrip('.')
        tag = f"safe_tail_keep{keep_topn}_add{add_topn}_min{min_str}"
        print("=" * 100)
        print(tag)

        sub_tmp, changed, count_dist = postprocess_lgbm_with_safe_tail_params(
            lgbm_submission_raw,
            keep_topn=keep_topn,
            add_topn=add_topn,
            min_rescue_score=min_score,
            top_k=BEST_TOP_K,
            verbose=True,
        )

        path = f"/kaggle/working/submission_friends_lgbm_{tag}.csv"
        sub_tmp.to_csv(path, index=False)
        sweep_outputs[tag] = sub_tmp

        sweep_summary.append({
            "tag": tag,
            "keep_topn": keep_topn,
            "add_topn": add_topn,
            "min_score": min_score,
            "changed_rows": len(changed),
            "count_dist": str(count_dist),
            "path": path,
        })

        print("saved:", path)

    sweep_df = pd.DataFrame(sweep_summary)
    sweep_df.to_csv("/kaggle/working/safe_tail_param_sweep_summary.csv", index=False)
    print("\nSafe-tail parameter sweep summary:")
    print(sweep_df.to_string(index=False))
    print("Saved summary: /kaggle/working/safe_tail_param_sweep_summary.csv")

    # ============================================================
    # FIX: Find the correct tag for (23,2,0.70) dynamically
    # ============================================================
    main_tag = None
    for entry in sweep_summary:
        if entry["keep_topn"] == 23 and entry["add_topn"] == 2 and entry["min_score"] == 0.70:
            main_tag = entry["tag"]
            break

    if main_tag is None:
        # fallback: use the first available safe-tail variant
        if sweep_outputs:
            main_tag = list(sweep_outputs.keys())[0]
            print(f"Warning: (23,2,0.70) tag not found. Using first available: {main_tag}")
        else:
            main_tag = None

    if main_tag and main_tag in sweep_outputs:
        lgbm_submission = sweep_outputs[main_tag].copy()
    else:
        print("Warning: No safe-tail submission found, using raw LGBM output.")
        lgbm_submission = lgbm_submission_raw.copy()

    lgbm_submission.to_csv("/kaggle/working/submission.csv", index=False)
    print("\nMain submission saved: /kaggle/working/submission.csv")
    print("Main tag:", main_tag if main_tag else "raw LGBM")
    print(lgbm_submission.head().to_string())

    # Optional diagnostic diff against raw LGBM and a stricter variant.
    if main_tag and main_tag in sweep_outputs and "safe_tail_keep24_add1_min075" in sweep_outputs:
        diff_23_2_vs_24_1 = compare_submissions(
            sweep_outputs[main_tag],
            sweep_outputs["safe_tail_keep24_add1_min075"],
            "k23_a2_070",
            "k24_a1_075",
            top_k=BEST_TOP_K,
        )
        diff_23_2_vs_24_1.to_csv("/kaggle/working/diff_23_2_070_vs_24_1_075.csv", index=False)
        print("Saved diff: /kaggle/working/diff_23_2_070_vs_24_1_075.csv")

else:
    print("LightGBM reranker branch disabled.")

Building LightGBM transfer maps...



Inference Embeddings: 100%|██████████| 36/36 [00:05<00:00,  6.23it/s]

Inference Embeddings: 100%|██████████| 36/36 [00:06<00:00,  5.85it/s]

Chunks: 100%|██████████| 2/2 [00:06<00:00,  3.39s/it]

Inference Embeddings: 100%|██████████| 38/38 [00:06<00:00,  6.26it/s]

Inference Embeddings: 100%|██████████| 38/38 [00:06<00:00,  5.76it/s]

Chunks: 100%|██████████| 2/2 [00:07<00:00,  3.61s/it]


Building LightGBM feature tables...


LGBM features train:   0%|          | 0/1139 [00:00<?, ?it/s]



Chunks: 100%|██████████| 1/1 [00:00<00:00, 25.22it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.30it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.35it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 13.80it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 13.54it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 13.66it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 24.84it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.51it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 27.96it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 13.78it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 24.05it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 27.24it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.91it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 24.58it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 24.47it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 13.90it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.69it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 14.

LGBM features val:   0%|          | 0/10 [00:00<?, ?it/s]



Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.77it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.33it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.78it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.55it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.82it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 16.21it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 14.24it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 16.58it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 22.94it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.73it/s]


LGBM features test:   0%|          | 0/40 [00:00<?, ?it/s]



Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.19it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.41it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.74it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 18.33it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.63it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.49it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.60it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.38it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 20.88it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 20.97it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 15.65it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.06it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.76it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.99it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.63it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.42it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 21.49it/s]


Chunks: 100%|██████████| 1/1 [00:00<00:00, 17.

Training LightGBM reranker...
[LightGBM] [Info] Number of positive: 4642, number of negative: 229921
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.011712 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 1290
[LightGBM] [Info] Number of data points in the train set: 234563, number of used features: 13
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Info] Start training from score 0.000000
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits

In [31]:

# ============================================================
# Generate submission
# ============================================================

if USE_LGBM_RERANKER_FOR_SUBMISSION and "lgbm_submission" in globals():
    submission = lgbm_submission.copy()
    submission["query_id"] = submission["query_id"].map(normalize_query_id)
    submission["predicted_citations"] = submission["predicted_citations"].map(clean_prediction_text)
    preds = submission["predicted_citations"].tolist()
    print("Using LightGBM reranker submission.")
else:
    preds = []
    for q in tqdm(test["query"].astype(str).tolist(), desc="Generating submission"):
        if USE_SUPERVISED_RERANKER_FOR_SUBMISSION:
            pred = retrieve_final_reranked(
                q,
                reranker=reranker_final_model,
                top_k=BEST_TOP_K,
                candidate_topn=RERANK_CANDIDATE_TOPN,
            )
        else:
            pred = retrieve_final(
                q,
                top_k=BEST_TOP_K,
                bge_topk=BEST_BGE_TOPK,
                dense_topn=BEST_DENSE_TOPN,
                dense_weight=BEST_DENSE_WEIGHT,
                lexical_weight=BEST_LEXICAL_WEIGHT,
                use_features=BEST_USE_FEATURES,
                feature_topn=BEST_FEATURE_TOPN,
                feature_weight=BEST_FEATURE_WEIGHT,
                use_citation_boost=BEST_USE_CITATION_BOOST,
                citation_boost_weight=BEST_CITATION_BOOST_WEIGHT,
                use_law_expansion=BEST_USE_LAW_EXPANSION,
                law_expansion_weight=BEST_LAW_EXPANSION_WEIGHT,
                use_safe_citation_rescue=BEST_USE_SAFE_CITATION_RESCUE,
                safe_rescue_weight=BEST_SAFE_RESCUE_WEIGHT,
                use_safe_verifier=BEST_USE_SAFE_VERIFIER,
                safe_verifier_weight=BEST_SAFE_VERIFIER_WEIGHT,
                use_safe_tail_replace=BEST_USE_SAFE_TAIL_REPLACE,
                safe_tail_keep_topn=BEST_SAFE_TAIL_KEEP_TOPN,
                safe_tail_add_topn=BEST_SAFE_TAIL_ADD_TOPN,
                safe_tail_min_score=BEST_SAFE_TAIL_MIN_SCORE,
            )
        preds.append(clean_prediction_text(";".join(pred)))
    submission = pd.DataFrame({
        "query_id": test["query_id"].map(normalize_query_id),
        "predicted_citations": preds,
    })


Using LightGBM reranker submission.


In [32]:
BEST_USE_LAW_FAMILY_RESCUE = True
BEST_LAW_FAMILY_TAIL_SLOTS = 1
BEST_LAW_FAMILY_TAIL_MIN_SCORE = 0.50

LAW_FAMILY_MANUAL_SCORE = 0.50
LAW_FAMILY_DYNAMIC_SCORE = 0.32
LAW_FAMILY_MIN_DYNAMIC_CNT = 2

LAW_FAMILY_MAP = {
    ("ZGB", "965"): [("ZGB", "656"), ("ZGB", "216"), ("ZGB", "839"), ("ZGB", "973"), ("ZGB", "968")],
    ("ZGB", "839"): [("ZGB", "965"), ("ZGB", "973"), ("ZGB", "968")],
    ("IPRG", "86"): [("IPRG", "17"), ("IPRG", "18"), ("IPRG", "90"), ("IPRG", "63")],
    ("OR", "267"): [("OR", "23"), ("OR", "127"), ("OR", "400"), ("OR", "959")],
    ("UVG", "9"): [("UVG", "2"), ("UVG", "6"), ("UVG", "36"), ("ATSG", "6")],
    ("UWG", "29"): [("UWG", "19"), ("UWG", "21"), ("UWG", "22"), ("MSchG", "42"), ("MSchG", "50b")],
}

def _lf_parse_law_article(cit):
    m = re.search(
        r"\bArt\.?\s*(\d+[a-zA-Z]?)"
        r"(?:\s*Abs\.?\s*[\d+a-zA-Z]+)?"
        r".*?\b(ZGB|OR|StGB|StPO|ZPO|BGG|SVG|SchKG|IPRG|AIG|ATSG|KVG|UVG|AHVG|IVG|URG|MSchG|UWG|MWSTG|DBG|VStG|ArG|BVG|PrHG|BV|EMRK)\b",
        str(cit),
        flags=re.IGNORECASE,
    )
    if not m:
        return None
    return (m.group(2).upper(), re.sub(r"\D", "", m.group(1)))

def _lf_find_matching_citations(target_law, target_art, candidate_pool):
    out = []
    for cit in candidate_pool:
        p = _lf_parse_law_article(cit)
        if not p:
            continue
        law, art = p
        if law == target_law.upper() and art == str(target_art):
            out.append(str(cit).strip())
    return out

def law_family_tail_rescue_scores(query, ranked_head, candidate_pool):
    if not BEST_USE_LAW_FAMILY_RESCUE:
        return Counter()

    scores = Counter()
    seed_pairs = set()

    if "extract_query_citations" in globals():
        for h in extract_query_citations(query):
            p = _lf_parse_law_article(h)
            if p:
                seed_pairs.add(p)

    for item in ranked_head[:10]:
        cit = item[0] if isinstance(item, (tuple, list)) else item
        p = _lf_parse_law_article(cit)
        if p:
            seed_pairs.add(p)

    for seed in seed_pairs:
        fams = LAW_FAMILY_MAP.get(seed, [])
        for law, art in fams:
            for cit in _lf_find_matching_citations(law, art, candidate_pool):
                scores[cit] += LAW_FAMILY_MANUAL_SCORE

    if "cit_cooccur" in globals():
        co_map = cit_cooccur
    elif "cooccur" in globals():
        co_map = cooccur
    elif "co_citation" in globals():
        co_map = co_citation
    elif "co_citation_counter" in globals():
        co_map = co_citation_counter
    else:
        co_map = {}

    for item in ranked_head[:8]:
        seed_cit = item[0] if isinstance(item, (tuple, list)) else item
        seed_cit = str(seed_cit).strip()

        for cc, cnt in sorted(co_map.get(seed_cit, {}).items(), key=lambda x: x[1], reverse=True)[:5]:
            if cnt < LAW_FAMILY_MIN_DYNAMIC_CNT:
                continue

            cc = str(cc).strip()
            if cc not in candidate_pool:
                continue

            scores[cc] += LAW_FAMILY_DYNAMIC_SCORE * min(float(cnt) / 5.0, 1.0)

    return scores

In [33]:

# Save final submission.
# If the LightGBM branch is enabled, `submission` was already built in the previous cell.
# Otherwise, it was built from `preds` using the original pipeline.
print(submission.head(10).to_string())
print("submission shape:", submission.shape)

submission.to_csv("/kaggle/working/submission.csv", index=False)
print("Saved to /kaggle/working/submission.csv")


   query_id                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                  predicted_citations
0  test_001                              Art. 82 Abs. 1 SchKG;Art. 271 Abs. 1 SchKG;Art. 55 Abs. 1 USG;Art. 12 Abs. 1 NHG;BGE 128 III 76 E. 1b;Art. 62 Abs. 1 URG;Art. 25 Abs. 1 BankG;Art. 965 Abs. 3 ZGB;Art. 960a Abs. 1 OR;Art. 655 Abs. 2 ZGB;Art. 8 ZGB;BGE 132 III 564 E. 6.2;Art. 121 OR;Art. 527 ZGB;Art. 965 Abs. 2 ZGB;1B_572/2021 E. 2.1;Art. 946 Abs. 1 ZGB;Art. 969 Abs. 1 ZGB;Art. 241 Abs. 3 ZGB;Art. 17 IPRG;BGE 148 V 21 E. 5.3;Art. 216 Abs. 2 ZGB;Art. 973 Abs. 1 ZGB;Art. 42 Abs. 